In [ ]:
from google.colab import drive
drive.mount ('/content/drive')
ROOT = '/content/drive/MyDrive/geoproject/'

In [ ]:
import warnings; warnings.filterwarnings('ignore')

In [ ]:
# ---------------------------------------------------------------- CELL AB0 --
# ABLATION STUDY on the final OSM pipeline -- post-hoc, NOT pre-registered.
# Scores OSM DEV, the 106 set and PK n>=3 only. The OSM TEST split is never read.
# Restore + gates. Nothing new is fitted here.
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, time, pickle
import numpy as np, pandas as pd
from collections import Counter
from scipy import sparse
from scipy.stats import binomtest
ROOT = '/content/drive/MyDrive/geoproject'; OUT = f'{ROOT}/osmfinal'
sys.path.insert(0, ROOT)
import pipeline_ranker as pr

D = pd.read_parquet(f'{OUT}/osm_extracted.parquet')
for c in ('adjectives', 'transaction', 'context_nouns'): D[c] = D[c].map(json.loads)
D['adjs'] = D['adjectives'].map(lambda L: sorted({a.lower() for a in L if str(a).isalpha()}))   # P1.1 adjs_of rule
D['pit']  = D['transaction'].map(lambda L: [it for it in L if it.startswith('amod:')])          # P1.1 amod_items
tr, dv = D[D.split == 'train'], D[D.split == 'dev']          # the test split is deliberately never bound

meta = np.load(f'{OUT}/osm_feat_meta.npz', allow_pickle=True)
PV, CLASSES, NADJ, MIN_SUP = list(meta['patvocab']), list(meta['classes']), int(meta['n_adj']), float(meta['min_support'])
CIX = {c: k for k, c in enumerate(CLASSES)}; K = len(CLASSES)
ytr, ydv = tr.tag.map(CIX).values, dv.tag.map(CIX).values
assert (ytr == meta['ytr']).all() and (ydv == meta['ydv']).all(), 'labels differ from R3'

def _ok(p):                                                  # P1.2 cleanup, verbatim
    body = p.split(':', 1)[1] if ':' in p else p
    return all(len(t) > 1 and t.isalnum() for t in body.split('_'))
def mine(train, min_sup):                                    # R3 support rule, parameterised by min_sup
    n_min = min_sup * len(train)
    c1 = Counter(a for L in train.adjs for a in L); c2 = Counter(p for L in train.pit for p in L)
    fa = sorted(a for a, c in c1.items() if c >= n_min and _ok(a))
    fp = sorted(p for p, c in c2.items() if c >= n_min and _ok(p))
    return fa + fp, len(fa)
def featurize(df, pv, nadj):                                 # R3 featurize, parameterised by vocabulary
    pix = {p: j for j, p in enumerate(pv)}; A, P2 = set(pv[:nadj]), set(pv[nadj:])
    ri, ci = [], []
    for r, row in enumerate(df.itertuples(index=False)):
        seen = {pix[a] for a in row.adjs if a in A} | {pix[p] for p in row.pit if p in P2}
        ri += [r] * len(seen); ci += list(seen)
    return sparse.csr_matrix((np.ones(len(ri), np.float32), (ri, ci)), shape=(len(df), len(pv)))
def assoc(Xt, yt, k):                                        # R4a counting, verbatim
    Cnt = np.zeros((Xt.shape[1], k)); r_, c_ = Xt.nonzero(); np.add.at(Cnt, (c_, yt[r_]), 1)
    Ntot = Cnt.sum(); rt = Cnt.sum(1, keepdims=True); ct = Cnt.sum(0, keepdims=True)
    conf = np.divide(Cnt, rt, out=np.zeros_like(Cnt), where=rt > 0)
    with np.errstate(divide='ignore', invalid='ignore'):
        lift = (Cnt / Ntot) / ((rt / Ntot) * (ct / Ntot))
    return conf.astype(np.float32), np.where(np.isfinite(lift) & (lift > 1), np.log(lift), 0.0).astype(np.float32)

# gate 1-3: the saved vocabulary, feature matrices and conf table are re-derived exactly
pv_, nadj_ = mine(tr, MIN_SUP); assert pv_ == PV and nadj_ == NADJ, 'R3 vocabulary not reproduced'
Xtr, Xdv = featurize(tr, PV, NADJ), featurize(dv, PV, NADJ)
for s, Xs in (('train', Xtr), ('dev', Xdv)):
    assert (Xs != sparse.load_npz(f'{OUT}/X_{s}.npz')).nnz == 0, f'X_{s} not reproduced'
T4 = dict(np.load(f'{OUT}/tables_r4a.npz'))
bz = np.load(f'{OUT}/tables_ben.npz'); T4['bayes_en'] = (bz['mean'] * bz['selected']).astype(np.float32)
CONF, LOGLIFT = assoc(Xtr, ytr, K)
assert np.allclose(CONF, T4['conf']) and np.allclose(LOGLIFT, T4['loglift']), 'R4a tables not reproduced'
print(f'gates 1-3 passed: {len(PV)} patterns ({NADJ} adj + {len(PV)-NADJ} pairs), X and tables identical')

LEX = json.load(open(f'{ROOT}/eval/evalset_106.json'))['lexicon']
WVT = pr.Tables(f'{ROOT}/results_i/i3_elasticnet.npz', f'{ROOT}/results_p/p1_bayesian_en.npz', f'{ROOT}/results_p/p1_feat_meta.npz')
RK = {'w2v':         pr.Ranker(pr.ladder(pickle.load(open(f'{OUT}/w2v_googlenews_osm.pkl', 'rb'))), LEX, WVT),
      'numberbatch': pr.Ranker(pr.ladder(pickle.load(open(f'{OUT}/numberbatch_osm.pkl', 'rb'))), LEX, WVT)}

def norm_rows(M):                                            # R5a, verbatim
    lo, hi = M.min(1, keepdims=True), M.max(1, keepdims=True)
    return np.where(hi > lo, (M - lo) / (hi - lo + 1e-9), M)
def sim_matrix(R, df, drop_abstract=True, extra=None):       # R5a sim_matrix; defaults = R5a exactly
    out = []
    for i, L in enumerate(df.context_nouns):
        terms = [t for t in L if not drop_abstract or t.lower() not in pr.ABSTRACT]
        if extra is not None: terms = terms + list(extra[i])
        out.append(R._norm(R.sim(terms)))
    return np.vstack(out).astype(np.float32)
def table_matrix(R, Xs, M, classes=CLASSES):                 # R5a, verbatim
    agg = Xs @ M
    S = np.zeros((Xs.shape[0], len(R.TAGS)), np.float32)
    for k, c in enumerate(classes):
        if c in R.IX: S[:, R.IX[c]] = agg[:, k]
    has_ = Xs.getnnz(1) > 0
    S[has_] = norm_rows(S[has_]); S[~has_] = 0.0
    return S
def gold_idx(R, df): return np.array([R.IX.get(t, -1) for t in df.tag])   # R7
def ranks(SIM, TBL, g, lam):                                               # R7
    s = SIM if TBL is None or lam == 0 else (1 - lam) * SIM + lam * TBL
    ok = g >= 0; r = np.full(len(g), SIM.shape[1] + 1)
    r[ok] = (s[ok] >= s[np.where(ok)[0], g[ok]][:, None]).sum(1); return r
def mets(r): return {'Hits@1': float((r <= 1).mean()), 'Hits@3': float((r <= 3).mean()), 'nDCG@20': float(np.where(r <= 20, 1 / np.log2(r + 1), 0).mean())}
LAMS = [round(x, 2) for x in np.arange(0.0, 0.85, 0.05)]
def pick(SIM, TBL, g):                                       # R5a lambda rule: Hits@3, then nDCG@20
    curve = {l: mets(ranks(SIM, TBL, g, l)) for l in LAMS}
    return max(LAMS, key=lambda l: (curve[l]['Hits@3'], curve[l]['nDCG@20']))
def boot(a, b, n=10000, seed=0):                             # R6, verbatim
    rs = np.random.RandomState(seed); m = len(a); d = np.empty(n); a = a.astype(float); b = b.astype(float)
    for i in range(n):
        ix = rs.randint(0, m, m); d[i] = a[ix].mean() - b[ix].mean()
    return float(a.mean() - b.mean()), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))

ev106 = json.load(open(f'{ROOT}/eval/evalset_106_extracted.json')); ev106 = ev106['rows'] if isinstance(ev106, dict) else ev106
evpk = json.load(open(f'{ROOT}/eval/pk_evalset_extracted.json')); evpk = evpk['rows'] if isinstance(evpk, dict) else evpk
cnt = Counter(r['gold_tag'] for r in evpk); pk3 = [r for r in evpk if cnt[r['gold_tag']] >= 3]
def as_df(rows):                                             # R7, verbatim
    return pd.DataFrame({'context_nouns': [r['context_nouns'] for r in rows], 'tag': [r['gold_tag'] for r in rows],
                         'adjs': [sorted({a.lower() for a in r.get('adjectives', []) if str(a).isalpha()}) for r in rows],
                         'pit': [[it for it in r.get('transaction', []) if it.startswith('amod:')] for r in rows],
                         'adjectives': [r.get('adjectives', []) for r in rows], 'transaction': [r.get('transaction', []) for r in rows]})
SETS = {'osm_dev': dv, 'eval_106': as_df(ev106), 'pk_n3': as_df(pk3)}
assert len(SETS['pk_n3']) == 157

ARMS, RANKS, SIMC = {}, {}, {}
def sims(bb, s, mode):
    k = (bb, s, mode)
    if k not in SIMC:
        R, df = RK[bb], SETS[s]
        SIMC[k] = (sim_matrix(R, df, drop_abstract=False) if mode == 'keep_abstract' else
                   sim_matrix(R, df, extra=list(df.adjectives)) if mode == 'naive' else sim_matrix(R, df))
    return SIMC[k]
def osm_table(pv, nadj, M, cols=None):
    def f(R, df):
        Xs = featurize(df, pv, nadj)
        return table_matrix(R, Xs if cols is None else Xs[:, cols], M if cols is None else M[cols])
    return f
def wv_table(M):                                             # Wikivoyage pattern table: S2 pat_table / Ranker.en_ben path
    def f(R, df):
        T = R.T; out = np.zeros((len(df), len(R.TAGS)), np.float32)
        for i, (a, t) in enumerate(zip(df.adjectives, df.transaction)):
            fx = T.pat_feats({'adjectives': a, 'transaction': t})
            if not fx: continue
            agg = M[fx].sum(0)
            for c, w in R.c2w_pat.items(): out[i, w] = agg[T.PCLS.index(c)]
            out[i] = R._norm(out[i])
        return out
    return f
def arm(name, bb, mode='nouns', tbl=None, sets=None):
    sets = sets or SETS; R = RK[bb]
    g = {s: gold_idx(R, sets[s]) for s in sets}
    TB = {s: (tbl(R, sets[s]) if tbl else None) for s in sets}
    S = {s: (sims(bb, s, mode) if sets is SETS else
             sim_matrix(R, sets[s], drop_abstract=(mode != 'keep_abstract'))) for s in sets}
    lam = pick(S['osm_dev'], TB['osm_dev'], g['osm_dev']) if tbl else 0.0
    RANKS[name] = {s: ranks(S[s], TB[s], g[s], lam) for s in sets}
    ARMS[name] = {'backbone': bb, 'similarity': mode, 'lambda': lam, **{s: mets(RANKS[name][s]) for s in sets}}
    print(f'{name:48s} lam {lam:.2f} | ' + ' | '.join(f"{s} H@3 {ARMS[name][s]['Hits@3']:.3f}" for s in sets))

# gate 4-5: final + base rows reproduce adoption.json (dev) and final.json (106, PK)
ADO, FIN = json.load(open(f'{OUT}/adoption.json')), json.load(open(f'{OUT}/final.json'))
arm('final: numberbatch + conf (657)', 'numberbatch', tbl=osm_table(PV, NADJ, T4['conf']))
arm('base Alg.2 + abstract removal (w2v)', 'w2v')
a, b = ARMS['final: numberbatch + conf (657)'], ARMS['base Alg.2 + abstract removal (w2v)']
assert a['lambda'] == ADO['dev']['numberbatch']['conf']['best_lambda'] and abs(a['osm_dev']['Hits@3'] - ADO['dev']['numberbatch']['conf']['Hits@3']) < 1e-9
assert abs(b['osm_dev']['Hits@3'] - ADO['dev']['w2v']['nouns']['Hits@3']) < 1e-9
for s in ('eval_106', 'pk_n3'):
    assert abs(a[s]['Hits@3'] - FIN['results'][s]['final: numberbatch + OSM conf']['Hits@3']) < 1e-9, s
    assert abs(b[s]['Hits@3'] - FIN['results'][s]['base paper (w2v, Algorithm 2)']['Hits@3']) < 1e-9, s
print('gates 4-5 passed: final and base rows reproduce adoption.json and final.json exactly')

In [ ]:
# ---------------------------------------------------------------- CELL AB1 --
# A. Stage-wise build-up from the base paper. B. Swaps at the final configuration.
cols_adj, cols_pair = np.arange(NADJ), np.arange(NADJ, len(PV))
arm('A0 base Alg.2, abstract nouns kept (w2v)',        'w2v', mode='keep_abstract')
arm('A2 + adjectives appended to the query (w2v)',     'w2v', mode='naive')
arm('A3 + adjective patterns only, conf (w2v)',        'w2v', tbl=osm_table(PV, NADJ, T4['conf'], cols_adj))
arm('A4 + adjective-noun pairs = 657 patterns (w2v)',  'w2v', tbl=osm_table(PV, NADJ, T4['conf']))
arm('B1 final, adjective patterns only',               'numberbatch', tbl=osm_table(PV, NADJ, T4['conf'], cols_adj))
arm('B2 final, pair patterns only',                    'numberbatch', tbl=osm_table(PV, NADJ, T4['conf'], cols_pair))
arm('B3 final, log-lift instead of confidence',        'numberbatch', tbl=osm_table(PV, NADJ, T4['loglift']))
arm('B4 final, frequentist EN (C=0.5)',                'numberbatch', tbl=osm_table(PV, NADJ, T4['fen_C0.5']))
arm('B5 final, Bayesian EN',                           'numberbatch', tbl=osm_table(PV, NADJ, T4['bayes_en']))
arm('B6 final, nouns only (no table)',                 'numberbatch')
arm('B7 final, adjectives appended instead of table',  'numberbatch', mode='naive')
assert abs(ARMS['A4 + adjective-noun pairs = 657 patterns (w2v)']['osm_dev']['Hits@3'] - ADO['dev']['w2v']['conf']['Hits@3']) < 1e-9
assert abs(ARMS['B5 final, Bayesian EN']['osm_dev']['Hits@3'] - ADO['dev']['numberbatch']['bayes_en']['Hits@3']) < 1e-9
assert abs(ARMS['B6 final, nouns only (no table)']['osm_dev']['Hits@3'] - ADO['dev']['numberbatch']['nouns']['Hits@3']) < 1e-9
print('A0 on the 106 set:', ARMS['A0 base Alg.2, abstract nouns kept (w2v)']['eval_106'], '(Run 1 reproduction: Hits@3 .415)')

# ---------------------------------------------------------------- CELL AB2 --
# C. Minimum-support sensitivity (no value is adopted from it).
SUPPORT = {}
for ms in (0.002, 0.001, 0.0005, 0.0002, 0.0001):
    pv_s, nadj_s = mine(tr, ms)
    conf_s, _ = assoc(featurize(tr, pv_s, nadj_s), ytr, K)
    name = f'C support {ms:g}'
    arm(name, 'numberbatch', tbl=osm_table(pv_s, nadj_s, conf_s))
    SUPPORT[name] = {'min_support': ms, 'patterns': len(pv_s), 'adjectives': nadj_s, 'pairs': len(pv_s) - nadj_s}
    print('   ', SUPPORT[name])
assert SUPPORT['C support 0.0005']['patterns'] == len(PV)

In [ ]:
# ---------------------------------------------------------------- CELL AB3 --
# D. Extractor: transformer parser (used throughout) vs spaCy small model. Same code, only the parser changes.
import subprocess, spacy
try: spacy.load('en_core_web_sm')
except OSError: subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)
import adjective_extractor as ae
I_sm = ae.PlaceInterpreter(model='en_core_web_sm')
sub = D[D.split.isin(['train', 'dev'])].reset_index(drop=True)
t0 = time.time(); rows_sm = pr.extract_rows(list(sub.text), I_sm, batch_size=256)
print(f'small-parser extraction: {len(rows_sm):,} texts in {time.time()-t0:.0f}s')
SM = pd.DataFrame({'split': sub.split, 'tag': sub.tag,
                   'context_nouns': [r['context_nouns'] for r in rows_sm],
                   'adjectives': [r['adjectives'] for r in rows_sm], 'transaction': [r['transaction'] for r in rows_sm]})
SM['adjs'] = SM['adjectives'].map(lambda L: sorted({a.lower() for a in L if str(a).isalpha()}))
SM['pit']  = SM['transaction'].map(lambda L: [it for it in L if it.startswith('amod:')])
sm_tr, sm_dv = SM[SM.split == 'train'], SM[SM.split == 'dev'].reset_index(drop=True)
assert (sm_dv.tag.values == dv.tag.values).all()
pv_sm, nadj_sm = mine(sm_tr, MIN_SUP)
conf_sm, _ = assoc(featurize(sm_tr, pv_sm, nadj_sm), sm_tr.tag.map(CIX).values, K)
def eval_sm(rows):
    txt = [r.get('sentence') or r.get('text') for r in rows]; assert all(txt), 'eval rows carry no sentence text'
    ex = pr.extract_rows(txt, I_sm)
    return as_df([{**r, 'adjectives': e['adjectives'], 'transaction': e['transaction']} for r, e in zip(rows, ex)])
SETS_SM = {'osm_dev': sm_dv, 'eval_106': eval_sm(ev106), 'pk_n3': eval_sm(pk3)}
arm('D final, small parser (en_core_web_sm)', 'numberbatch', tbl=osm_table(pv_sm, nadj_sm, conf_sm), sets=SETS_SM)
EXTRACTOR = {'patterns': len(pv_sm), 'adjectives': nadj_sm, 'pairs': len(pv_sm) - nadj_sm,
             'dev_rows_with_pattern': float((featurize(sm_dv, pv_sm, nadj_sm).getnnz(1) > 0).mean()),
             'dev_rows_with_pattern_trf': float((Xdv.getnnz(1) > 0).mean())}
print(EXTRACTOR)

In [ ]:
# ---------------------------------------------------------------- CELL AB4 --
# E. Guided vs extended FP-Growth on the OSM training transactions (min support 0.0005).
# Extended = FP-Growth's first scan (drop infrequent items) + mlxtend fpgrowth over every remaining item;
# both are inside the timed/measured block. Guided = count only {pattern, tag} itemsets.
import tracemalloc, signal, warnings; warnings.filterwarnings('ignore')
try: from mlxtend.frequent_patterns import fpgrowth
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mlxtend'], check=True); from mlxtend.frequent_patterns import fpgrowth
from mlxtend.preprocessing import TransactionEncoder
KEEP = ('amod:', 'compound:', 'type:')                        # Run 1's item types; ctx: nouns are not mined
TX = [sorted(set([f'adj:{a}' for a in A] + [it for it in T_ if it.startswith(KEEP)] + [f'tag:{y}']))
      for A, T_, y in zip(tr.adjs, tr.transaction, tr.tag)]
N_TX = len(TX); MIN_CNT = MIN_SUP * N_TX
is_pat = lambda it: it.startswith(('adj:', 'amod:'))
FPG = {'transactions': N_TX, 'min_support': MIN_SUP}
tracemalloc.start(); t0 = time.perf_counter()               # guided
c1, c12 = Counter(), Counter()
for items in TX:
    tag = next(i for i in items if i.startswith('tag:'))
    for it in items:
        if is_pat(it): c1[it] += 1; c12[(it, tag)] += 1
G = {k for k, c in c12.items() if c >= MIN_CNT and c1[k[0]] >= MIN_CNT}
FPG['guided'] = {'seconds': time.perf_counter() - t0, 'peak_mb': tracemalloc.get_traced_memory()[1] / 2**20, 'pattern_tag_itemsets': len(G)}
tracemalloc.stop()
class _Timeout(Exception): pass
def _alarm(*_): raise _Timeout()
signal.signal(signal.SIGALRM, _alarm)
tracemalloc.start(); t0 = time.perf_counter(); signal.alarm(20 * 60)
try:
    ic = Counter(it for items in TX for it in items)                        # FP-Growth first scan
    fset = {it for it, c in ic.items() if c >= MIN_CNT}
    TXf = [[it for it in items if it in fset] for items in TX]
    te_ = TransactionEncoder(); DF_ = pd.DataFrame(te_.fit(TXf).transform(TXf), columns=te_.columns_)
    FI = fpgrowth(DF_, min_support=MIN_SUP, use_colnames=True); signal.alarm(0)
    E = set()
    for its in FI.itemsets:
        if len(its) == 2:
            a_, b_ = sorted(its, key=lambda x: x.startswith('tag:'))
            if is_pat(a_) and b_.startswith('tag:'): E.add((a_, b_))
    FPG['extended'] = {'seconds': time.perf_counter() - t0, 'peak_mb': tracemalloc.get_traced_memory()[1] / 2**20,
                       'frequent_items': len(fset), 'frequent_itemsets': int(len(FI)), 'pattern_tag_itemsets': len(E)}
    FPG['identical_pattern_tag_itemsets'] = (E == G)
    FPG['speedup'] = FPG['extended']['seconds'] / max(FPG['guided']['seconds'], 1e-9)
except _Timeout:
    FPG['extended'] = {'did_not_finish_within_s': 20 * 60}
finally:
    signal.alarm(0); tracemalloc.stop()
print(json.dumps(FPG, indent=1))

In [ ]:
# ---------------------------------------------------------------- CELL AB5 --
# F. Matched corpus comparison: same backbone, same table type on mined patterns; only the corpus differs.
wl = pd.read_parquet(f'{ROOT}/results_i/weak_labelled.parquet', columns=['transaction', 'pairs', 'weak_tag', 'split'])
wl = wl[wl.split == 'train'].copy()
wl['transaction'] = wl['transaction'].map(json.loads)
wl['adjectives']  = wl['pairs'].map(lambda s: sorted({p[0] for p in json.loads(s) if p[2] == 'amod'}))
assert len(wl) == 398046
wcix = {c: j for j, c in enumerate(WVT.PCLS)}; Cw = np.zeros((len(WVT.PV), len(WVT.PCLS)))
for a, t, y in zip(wl.adjectives, wl.transaction, wl.weak_tag):
    j = wcix.get(y)
    if j is None: continue
    for i in set(WVT.pat_feats({'adjectives': a, 'transaction': t})): Cw[i, j] += 1
print('WV counts:', int((Cw > 0).sum()), int(Cw.sum()), '(S1 run: 40379, 898871)')
assert (Cw > 0).sum() == 40379 and int(Cw.sum()) == 898871, 'Wikivoyage pattern counts differ from the S1 run'
rtw = Cw.sum(1, keepdims=True); WV_CONF = np.divide(Cw, rtw, out=np.zeros_like(Cw), where=rtw > 0).astype(np.float32)
arm('F Wikivoyage conf (966 patterns)',   'numberbatch', tbl=wv_table(WV_CONF))
arm('F Wikivoyage Bayesian EN (966)',     'numberbatch', tbl=wv_table((WVT.MU * WVT.SEL).astype(np.float32)))
assert abs(ARMS['F Wikivoyage Bayesian EN (966)']['osm_dev']['Hits@3'] - ADO['dev']['numberbatch']['wv_ben']['Hits@3']) < 1e-9

In [ ]:
# ---------------------------------------------------------------- CELL AB6 --
# Comparisons (paired bootstrap on Hits@3 + McNemar exact), Holm across the whole ablation family.
F_, A1 = 'final: numberbatch + conf (657)', 'base Alg.2 + abstract removal (w2v)'
PAIRS = [('A1 abstract-noun removal',         A1, 'A0 base Alg.2, abstract nouns kept (w2v)'),
         ('A2 adjectives appended to query',  'A2 + adjectives appended to the query (w2v)', A1),
         ('A3 adjective patterns (table)',    'A3 + adjective patterns only, conf (w2v)', A1),
         ('A4 + pair patterns',               'A4 + adjective-noun pairs = 657 patterns (w2v)', 'A3 + adjective patterns only, conf (w2v)'),
         ('A5 Numberbatch backbone',          F_, 'A4 + adjective-noun pairs = 657 patterns (w2v)'),
         ('A total: final vs base Alg.2',     F_, 'A0 base Alg.2, abstract nouns kept (w2v)'),
         ('B1 drop pair patterns',            'B1 final, adjective patterns only', F_),
         ('B2 drop adjective patterns',       'B2 final, pair patterns only', F_),
         ('B3 log-lift weighting',            'B3 final, log-lift instead of confidence', F_),
         ('B4 frequentist EN selection',      'B4 final, frequentist EN (C=0.5)', F_),
         ('B5 Bayesian EN selection',         'B5 final, Bayesian EN', F_),
         ('B6 drop the table',                'B6 final, nouns only (no table)', F_),
         ('B7 naive adjectives, no table',    'B7 final, adjectives appended instead of table', F_),
         ('D small parser',                   'D final, small parser (en_core_web_sm)', F_),
         ('F corpus, conf',                   F_, 'F Wikivoyage conf (966 patterns)'),
         ('F corpus, Bayesian EN',            'B5 final, Bayesian EN', 'F Wikivoyage Bayesian EN (966)')]
PAIRS += [(f'C {n[2:]} vs 0.0005', n, 'C support 0.0005') for n in SUPPORT if n != 'C support 0.0005']
CMP = []
for label, a, b in PAIRS:
    for s in SETS:
        ha, hb = RANKS[a][s] <= 3, RANKS[b][s] <= 3
        assert len(ha) == len(hb)
        d, lo, hi = boot(ha, hb); n01, n10 = int((~ha & hb).sum()), int((ha & ~hb).sum())
        p = binomtest(min(n01, n10), n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
        CMP.append({'comparison': label, 'set': s, 'a': a, 'b': b, 'delta_hits3': d, 'ci95': [lo, hi],
                    'a_only': n10, 'b_only': n01, 'p': float(p)})
ps = np.array([c['p'] for c in CMP]); o = np.argsort(ps); m_ = len(ps); adj = np.empty(m_); run = 0.0
for rank_, i in enumerate(o): run = max(run, min(1.0, (m_ - rank_) * ps[i])); adj[i] = run
for c, pa in zip(CMP, adj): c['p_holm'] = float(pa)
for c in CMP:
    print(f"{c['comparison']:30s} {c['set']:9s} dH@3 {c['delta_hits3']:+.3f} [{c['ci95'][0]:+.3f},{c['ci95'][1]:+.3f}] p_holm {c['p_holm']:.4f}")
json.dump({'status': 'post-hoc ablation, not pre-registered; the OSM test split was not read',
           'lambda_rule': 'per arm, chosen on OSM dev by Hits@3 then nDCG@20 (R5a); dev metrics are at that lambda, '
                          'so optimistic for every arm alike; 106 and PK n>=3 use the dev-chosen lambda unchanged',
           'sets': {s: len(v) for s, v in SETS.items()}, 'arms': ARMS, 'support_sensitivity': SUPPORT,
           'extractor': EXTRACTOR, 'fpgrowth': FPG, 'comparisons': CMP, 'holm_family_size': m_},
          open(f'{OUT}/ablation.json', 'w'), indent=1, default=str)
print('\nwrote', f'{OUT}/ablation.json')

In [ ]:
# ---------------------------------------------------------------- CELL AB7 --
# Guided vs extended FP-Growth on OSM train: identical (pattern, tag) itemsets? time, memory.
# Subsamples 5k/10k/20k/40k (seed 0) + the full 55,897 with NO time cap. Self-contained (fresh kernel).
# Writes osmfinal/fpgrowth_check.json only.
import warnings; warnings.filterwarnings('ignore')
from google.colab import drive; drive.mount('/content/drive')
import os, sys, json, time, subprocess, tracemalloc
import numpy as np, pandas as pd
from collections import Counter
try: from mlxtend.frequent_patterns import fpgrowth
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mlxtend'], check=True); from mlxtend.frequent_patterns import fpgrowth
from mlxtend.preprocessing import TransactionEncoder
ROOT = '/content/drive/MyDrive/geoproject'; OUT = f'{ROOT}/osmfinal'
D = pd.read_parquet(f'{OUT}/osm_extracted.parquet', columns=['adjectives', 'transaction', 'tag', 'split'])
D = D[D.split == 'train']
for c in ('adjectives', 'transaction'): D[c] = D[c].map(json.loads)
adjs = D['adjectives'].map(lambda L: sorted({a.lower() for a in L if str(a).isalpha()}))
KEEP = ('amod:', 'compound:', 'type:')
TX_ALL = [sorted(set([f'adj:{a}' for a in A] + [it for it in T_ if it.startswith(KEEP)] + [f'tag:{y}']))
          for A, T_, y in zip(adjs, D.transaction, D.tag)]
MIN_SUP = float(np.load(f'{OUT}/osm_feat_meta.npz', allow_pickle=True)['min_support'])
is_pat = lambda it: it.startswith(('adj:', 'amod:'))

def guided(TX):
    n_min = MIN_SUP * len(TX); tracemalloc.start(); t0 = time.perf_counter()
    c1, c12 = Counter(), Counter()
    for items in TX:
        tag = next(i for i in items if i.startswith('tag:'))
        for it in items:
            if is_pat(it): c1[it] += 1; c12[(it, tag)] += 1
    G = {k for k, c in c12.items() if c >= n_min and c1[k[0]] >= n_min}
    out = {'seconds': time.perf_counter() - t0, 'peak_mb': tracemalloc.get_traced_memory()[1] / 2**20, 'pattern_tag_itemsets': len(G)}
    tracemalloc.stop(); return G, out
def extended(TX):
    n_min = MIN_SUP * len(TX); tracemalloc.start(); t0 = time.perf_counter()
    ic = Counter(it for items in TX for it in items)                      # FP-Growth first scan (timed)
    fset = {it for it, c in ic.items() if c >= n_min}
    TXf = [[it for it in items if it in fset] for items in TX]
    te = TransactionEncoder(); DF = pd.DataFrame(te.fit(TXf).transform(TXf), columns=te.columns_)
    FI = fpgrowth(DF, min_support=MIN_SUP, use_colnames=True)
    E = set()
    for its in FI.itemsets:
        if len(its) == 2:
            a_, b_ = sorted(its, key=lambda x: x.startswith('tag:'))
            if is_pat(a_) and b_.startswith('tag:'): E.add((a_, b_))
    out = {'seconds': time.perf_counter() - t0, 'peak_mb': tracemalloc.get_traced_memory()[1] / 2**20,
           'frequent_items': len(fset), 'frequent_itemsets': int(len(FI)), 'pattern_tag_itemsets': len(E)}
    tracemalloc.stop(); return E, out

rs = np.random.RandomState(0); RES = {'min_support': MIN_SUP, 'runs': []}
for n in (5000, 10000, 20000, 40000, len(TX_ALL)):
    TX = TX_ALL if n == len(TX_ALL) else [TX_ALL[i] for i in rs.choice(len(TX_ALL), n, replace=False)]
    G, g = guided(TX); E, e = extended(TX)
    row = {'transactions': n, 'guided': g, 'extended': e, 'identical': E == G,
           'only_guided': len(G - E), 'only_extended': len(E - G), 'speedup': e['seconds'] / max(g['seconds'], 1e-9)}
    RES['runs'].append(row); print(json.dumps(row))
    json.dump(RES, open(f'{OUT}/fpgrowth_check.json', 'w'), indent=1)       # saved after every size
print('wrote', f'{OUT}/fpgrowth_check.json')

In [ ]:
%%writefile /content/build_docx.py
"""Build the thesis .docx from the markdown draft and the Air University template.
usage: python build_docx.py DRAFT.md TEMPLATE.docx ASSET_ROOT OUT.docx"""
import re, sys, os, zipfile, shutil, tempfile
from xml.sax.saxutils import escape
import pandas as pd, pypandoc

DRAFT, TEMPLATE, ROOT, OUT = sys.argv[1:5]
md = open(DRAFT, encoding='utf-8').read()
missing = []

def raw(xml): return f'\n```{{=openxml}}\n{xml}\n```\n'
PAGE = raw('<w:p><w:r><w:br w:type="page"/></w:r></w:p>')
def centred(text, size=24, bold=False):
    b = '<w:b/>' if bold else ''
    return raw(f'<w:p><w:pPr><w:jc w:val="center"/><w:spacing w:after="200"/></w:pPr><w:r><w:rPr>{b}'
               f'<w:sz w:val="{size}"/></w:rPr><w:t xml:space="preserve">{escape(text)}</w:t></w:r></w:p>')
def field(instr, placeholder='Right-click and choose Update Field.'):
    return raw('<w:p><w:r><w:fldChar w:fldCharType="begin" w:dirty="true"/></w:r>'
               f'<w:r><w:instrText xml:space="preserve"> {instr} </w:instrText></w:r>'
               '<w:r><w:fldChar w:fldCharType="separate"/></w:r>'
               f'<w:r><w:t>{placeholder}</w:t></w:r><w:r><w:fldChar w:fldCharType="end"/></w:r></w:p>')
def eq_table(expr, num):
    nb = ''.join(f'<w:{s} w:val="nil"/>' for s in ('top', 'left', 'bottom', 'right', 'insideH', 'insideV'))
    cell = lambda pct, jc, t: (f'<w:tc><w:tcPr><w:tcW w:w="{pct}" w:type="pct"/><w:vAlign w:val="center"/></w:tcPr>'
                               f'<w:p><w:pPr><w:jc w:val="{jc}"/></w:pPr><w:r><w:t xml:space="preserve">{escape(t)}</w:t></w:r></w:p></w:tc>')
    return raw(f'<w:tbl><w:tblPr><w:tblW w:w="5000" w:type="pct"/><w:tblBorders>{nb}</w:tblBorders></w:tblPr>'
               '<w:tblGrid><w:gridCol w:w="936"/><w:gridCol w:w="7488"/><w:gridCol w:w="936"/></w:tblGrid><w:tr>'
               + cell(500, 'left', '') + cell(4000, 'center', expr) + cell(500, 'right', f'({num})') + '</w:tr></w:tbl>')
def csv_table(path):
    full = os.path.join(ROOT, path)
    if not os.path.exists(full):
        missing.append(path); return f'\n**[MISSING TABLE: {path}]**\n'
    df = pd.read_csv(full).fillna('')
    if path.endswith('tab_3-12_hyperparameters.csv'):
        df = df.replace('0.0005 → 582 adjective 1-itemsets + 385 adjective–noun 2-itemsets',
                        '0.0005 → 582 adjective 1-itemsets + 384 adjective–noun 2-itemsets')
    for c in df.columns:
        if df[c].dtype == float and (df[c].dropna() % 1 != 0).any(): df[c] = df[c].map(lambda v: f'{v:.3f}' if v != '' else v)
    cellfix = lambda v: str(v).replace('|', '/').replace('\n', ' ')
    head = '| ' + ' | '.join(cellfix(c) for c in df.columns) + ' |'
    sep = '|' + '|'.join('---' for _ in df.columns) + '|'
    rows = ['| ' + ' | '.join(cellfix(v) for v in r) + ' |' for r in df.itertuples(index=False)]
    return '\n' + '\n'.join([head, sep] + rows) + '\n'
def fig(path):
    full = os.path.join(ROOT, path)
    if not os.path.exists(full):
        missing.append(path); return f'\n**[MISSING FIGURE: {path}]**\n'
    return f'\n![]({full}){{width=15cm}}\n'

title = md.split('\n', 1)[0].lstrip('# ').strip()
body = md[md.index('## Abstract'):]
abstract, rest = body.split('## Symbols and Abbreviations', 1)
nomen, chapters = rest.split('# Chapter 1', 1)
chapters = '# Chapter 1' + chapters
abstract = abstract.replace('## Abstract', '').replace('\n---\n', '\n')
nomen = re.sub(r'(?m)^---\s*$', '', nomen)
B = '[to be completed]'
front = ''.join([
    centred(title, 48, True), centred('by', 28), centred('[Student full name]', 36),
    centred('[Previous degree, university, year]', 24),
    centred('A thesis submitted in partial fulfillment of the requirements for the degree of', 32),
    centred('[Master of Science in Artificial Intelligence]', 32), centred('AIR UNIVERSITY', 32, True),
    centred('[Year]', 32), PAGE,
    centred(title, 48, True), centred('by', 28), centred('[Student full name]', 36),
    centred('A thesis submitted in partial fulfillment of the requirements for the degree of', 32),
    centred('[Master of Science in Artificial Intelligence]', 32), centred('SUPERVISOR', 28, True),
    centred('[Supervisor name and title]', 28), centred('AIR UNIVERSITY', 32, True), centred('ISLAMABAD', 32, True),
    centred('© [Author full name] [Year]. All rights reserved.', 24), PAGE,
    '\n##### Certificate\n\n', centred('[Department name]', 28),
    f'\nIt is hereby certified that [Student full name] ([Registration number]) has successfully completed his thesis.\n\n',
    *[f'\n{B}  \n[Name, title]  \nAir University  \n{r}\n' for r in ('Supervisor', 'Co-Supervisor', 'Internal Examiner, Guidance and Evaluation Committee',
      'External Examiner, Guidance and Evaluation Committee', 'Chair, [Department name]', 'Dean, [Faculty name]', 'Dean, Faculty of Graduate Studies')],
    PAGE, '\n##### Declaration\n\n',
    'I declare that all material in this thesis is my own work, and that which is not my own work has been identified and '
    'acknowledged. No material in this thesis has previously been submitted for a degree at this or any other university.\n\n'
    'Signature: ____________  \nAuthor\'s Name: [Student full name]  \nDated: ____________\n\n'
    'It is certified that the work presented in this thesis was carried out and completed under my supervision.\n\n'
    'Signature: ____________  \nSupervisor\'s Name: [Supervisor name]  \n[Title]  \n[Department name]  \nAir University, Islamabad  \nDated: ____________\n',
    PAGE, '\n##### Publications\n\n[List journal and conference publications, or state "None".]\n',
    PAGE, '\n##### Acknowledgments\n\n[Acknowledgments to be written by the author.]\n',
    PAGE, '\n##### Abstract\n', abstract,
    '\nThesis Supervisor: [Supervisor name]  \nTitle: [Supervisor title]\n',
    PAGE, '\n##### Table of Contents\n', field(r'TOC \o "1-3" \h \z \u \t "Heading 5,1"'),
    PAGE, '\n##### List of Figures\n', field(r'TOC \h \z \t "Figure Caption,1"'),
    PAGE, '\n##### List of Tables\n', field(r'TOC \h \z \t "Table Caption,1"'),
    PAGE, '\n##### Symbols and Abbreviations\n', nomen,
    raw('<w:p><w:r><w:t>@@SECTBREAK@@</w:t></w:r></w:p>')])

t = chapters
t = re.sub(r'(?m)^# (References|Appendix.*)$', r'##### \1', t)
t = re.sub(r'(?m)^# Chapter (\d+) — (.+)$', lambda m: (PAGE if m.group(1) != '1' else '') + f'# Chapter {m.group(1)}: {m.group(2)}', t)
t = re.sub(r'(?m)^##### References', PAGE.strip('\n') + '\n\n##### References', t)
t = re.sub(r'(?m)^---\s*$', '', t)
t = re.sub(r'<<FIG ([^>]+)>>', lambda m: fig(m.group(1).strip()), t)
t = re.sub(r'<<TABLE ([^>]+)>>', lambda m: csv_table(m.group(1).strip()), t)
t = re.sub(r'<<EQ ([\d.]+)\|(.+?)>>', lambda m: eq_table(m.group(2), m.group(1)), t)
def caption(m):
    kind = 'Figure Caption' if m.group(1) == 'Figure' else 'Table Caption'
    return f'\n::: {{custom-style="{kind}"}}\n{m.group(1)} {m.group(2)}\n:::\n'
t = re.sub(r'(?ms)^\*(Figure|Table) (\d+\.\d+ — .+?)\*\s*$', caption, t)
full = front + '\n' + t

def xml_table(lines):
    rows = [[c.strip() for c in l.strip().strip('|').split('|')] for l in lines if not re.match(r'^\|[\s:|-]+\|$', l.strip())]
    ncol = max(len(r) for r in rows)
    bd = ''.join(f'<w:{b} w:val="single" w:sz="4" w:space="0" w:color="000000"/>' for b in ('top', 'left', 'bottom', 'right', 'insideH', 'insideV'))
    def run(txt, bold):
        parts = re.split(r'(\*\*[^*]+\*\*|\*[^*]+\*)', txt); out = ''
        for part in parts:
            if not part: continue
            b_ = bold or part.startswith('**'); i_ = part.startswith('*') and not part.startswith('**')
            part = part.strip('*')
            rpr = ('<w:b/>' if b_ else '') + ('<w:i/>' if i_ else '') + '<w:sz w:val="20"/>'
            out += f'<w:r><w:rPr>{rpr}</w:rPr><w:t xml:space="preserve">{escape(part)}</w:t></w:r>'
        return out
    x = f'<w:tbl><w:tblPr><w:tblW w:w="5000" w:type="pct"/><w:tblBorders>{bd}</w:tblBorders><w:tblLayout w:type="autofit"/></w:tblPr>'
    x += '<w:tblGrid>' + ''.join(f'<w:gridCol w:w="{9360 // ncol}"/>' for _ in range(ncol)) + '</w:tblGrid>'
    for k, r in enumerate(rows):
        r = r + [''] * (ncol - len(r))
        x += '<w:tr>' + ('<w:trPr><w:tblHeader/></w:trPr>' if k == 0 else '')
        x += ''.join(f'<w:tc><w:tcPr><w:tcW w:w="{5000 // ncol}" w:type="pct"/></w:tcPr><w:p><w:pPr><w:spacing w:before="20" w:after="20" w:line="240" w:lineRule="auto"/></w:pPr>{run(c, k == 0)}</w:p></w:tc>' for c in r)
        x += '</w:tr>'
    return raw(x + '</w:tbl>')
out, buf = [], []
for line in full.split('\n'):
    if line.lstrip().startswith('|') and line.rstrip().endswith('|'):
        buf.append(line); continue
    if buf: out.append(xml_table(buf)); buf = []
    out.append(line)
if buf: out.append(xml_table(buf))
full = '\n'.join(out)

tmp = tempfile.mkdtemp()
pypandoc.convert_text(full, 'docx', format='markdown+raw_attribute+fenced_divs',
                      outputfile=os.path.join(tmp, 'body.docx'), extra_args=[f'--reference-doc={TEMPLATE}'])

src = os.path.join(tmp, 'body.docx'); zin = zipfile.ZipFile(src); zout = zipfile.ZipFile(OUT, 'w', zipfile.ZIP_DEFLATED)
for item in zin.infolist():
    data = zin.read(item.filename)
    if item.filename == 'word/styles.xml':
        x = data.decode('utf-8')
        for h in ('Heading1', 'Heading2', 'Heading3', 'Heading4', 'Heading5'):
            x = re.sub(rf'(<w:style [^>]*w:styleId="{h}".*?</w:style>)', lambda m: re.sub(r'<w:numPr>.*?</w:numPr>', '', m.group(1), flags=re.S), x, flags=re.S)
        for cid in ('FigureCaption', 'TableCaption'):
            x = re.sub(rf'(<w:style [^>]*w:styleId="{cid}"[^>]*>.*?)<w:basedOn w:val="[^"]+"/>', r'\1<w:basedOn w:val="Caption"/>', x, flags=re.S)
            if f'w:styleId="{cid}"' in x and '<w:basedOn w:val="Caption"/>' not in re.search(rf'<w:style [^>]*w:styleId="{cid}".*?</w:style>', x, re.S).group(0):
                x = re.sub(rf'(<w:style [^>]*w:styleId="{cid}"[^>]*>\s*<w:name [^>]*/>)', r'\1<w:basedOn w:val="Caption"/>', x)
        data = x.encode('utf-8')
    if item.filename == 'word/document.xml':
        x = data.decode('utf-8')
        bd = ''.join(f'<w:{b} w:val="single" w:sz="4" w:space="0" w:color="000000"/>' for b in ('top', 'left', 'bottom', 'right', 'insideH', 'insideV'))
        def fix_tbl(m):
            pr = m.group(0)
            if 'w:val="nil"' in pr: return pr
            pr = re.sub(r'<w:tblW [^>]*/>', '', pr); pr = re.sub(r'<w:tblBorders>.*?</w:tblBorders>', '', pr, flags=re.S)
            pr = re.sub(r'<w:tblLayout [^>]*/>', '', pr)
            ins = f'<w:tblW w:w="5000" w:type="pct"/><w:tblBorders>{bd}</w:tblBorders><w:tblLayout w:type="autofit"/>'
            m2 = re.search(r'<w:tblStyle [^>]*/>', pr)
            return pr[:m2.end()] + ins + pr[m2.end():] if m2 else pr.replace('<w:tblPr>', '<w:tblPr>' + ins, 1)
        x = re.sub(r'<w:tblPr>.*?</w:tblPr>', fix_tbl, x, flags=re.S)
        last = re.findall(r'<w:sectPr[ >].*?</w:sectPr>', x, re.S)[-1]
        def sect(fmt):
            y = re.sub(r'<w:pgNumType[^>]*/>', '', last)
            y = re.sub(r'<w:type w:val="[^"]*"\s*/>', '<w:type w:val="nextPage"/>', y)
            return re.sub(r'(<w:cols\b)', f'<w:pgNumType w:fmt="{fmt}" w:start="1"/>\\1', y, count=1)
        front_sect, body_sect = sect('lowerRoman'), sect('decimal')
        x = x.replace(last, body_sect)
        x = re.sub(r'<w:p>(?:<w:pPr>.*?</w:pPr>)?<w:r>(?:<w:rPr>.*?</w:rPr>)?<w:t[^>]*>@@SECTBREAK@@</w:t></w:r></w:p>',
                   f'<w:p><w:pPr>{front_sect}</w:pPr></w:p>', x, flags=re.S)
        assert '@@SECTBREAK@@' not in x
        data = x.encode('utf-8')
    if re.match(r'word/header\d*\.xml', item.filename):
        x = data.decode('utf-8')
        x = re.sub(r'<w:t([^>]*)>[^<]*</w:t>', r'<w:t\1></w:t>', x)
        data = x.encode('utf-8')
    if item.filename == 'word/settings.xml':
        x = data.decode('utf-8')
        if 'w:updateFields' not in x: x = x.replace('</w:settings>', '<w:updateFields w:val="true"/></w:settings>')
        data = x.encode('utf-8')
    zout.writestr(item, data)
zout.close(); shutil.rmtree(tmp)
print('wrote', OUT)
print('missing assets:', len(missing)); [print('  ', m) for m in missing]

In [ ]:
import subprocess, sys, glob, os
from google.colab import drive; drive.mount('/content/drive')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pypandoc_binary'], check=True)
G = '/content/drive/MyDrive/geoproject'
tpl = glob.glob('/content/drive/MyDrive/**/MS_Thesis_Template.docx', recursive=True); print('template:', tpl)
drafts = {os.path.basename(p): p for p in glob.glob(f'{G}/**/finalosmdraft*.md', recursive=True)}; print('drafts:', drafts)
os.makedirs(f'{G}/thesis_output', exist_ok=True)
for name, out in (('finalosmdraft.md', 'Thesis_APA.docx'), ('finalosmdraft_IEEE.md', 'Thesis_IEEE.docx')):
    r = subprocess.run([sys.executable, '/content/build_docx.py', drafts[name], tpl[0], G, f'{G}/thesis_output/{out}'], capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])

In [ ]:
# patch: replace the slow section-break regex with a direct string search
p = '/content/build_docx.py'; s = open(p).read()
a = """        x = re.sub(r'<w:p>(?:<w:pPr>.*?</w:pPr>)?<w:r>(?:<w:rPr>.*?</w:rPr>)?<w:t[^>]*>@@SECTBREAK@@</w:t></w:r></w:p>',
                   f'<w:p><w:pPr>{front_sect}</w:pPr></w:p>', x, flags=re.S)"""
b = """        i = x.index('@@SECTBREAK@@'); a0 = max(x.rfind('<w:p>', 0, i), x.rfind('<w:p ', 0, i)); a1 = x.index('</w:p>', i) + 6
        x = x[:a0] + f'<w:p><w:pPr>{front_sect}</w:pPr></w:p>' + x[a1:]"""
assert s.count(a) == 1; open(p, 'w').write(s.replace(a, b)); print('patched')

In [ ]:
# patch 2: appendix tables (Table A.1, B.1) are captioned tables; each appendix starts on a new page
p = '/content/build_docx.py'; s = open(p).read()
a = r"""t = re.sub(r'(?ms)^\*(Figure|Table) (\d+\.\d+ — .+?)\*\s*$', caption, t)"""
b = r"""t = re.sub(r'(?ms)^\*(Figure|Table) ([A-Z0-9]+\.\d+ — .+?)\*\s*$', caption, t)"""
assert s.count(a) == 1, 'caption line not found'; s = s.replace(a, b)
a = """t = re.sub(r'(?m)^##### References', PAGE.strip('\\n') + '\\n\\n##### References', t)"""
b = a + """\nt = re.sub(r'(?m)^##### (Appendix)', PAGE.strip('\\n') + '\\n\\n##### \\\\1', t)"""
assert s.count(a) == 1, 'references line not found'; s = s.replace(a, b)
open(p, 'w').write(s); print('patched 2')

In [ ]:
import hashlib; print(hashlib.md5(open('/content/build_docx.py','rb').read()).hexdigest())

In [ ]:
import tokenize, io, hashlib
src = open('/content/build_docx.py').read(); out = []
for tok in tokenize.generate_tokens(io.StringIO(src).readline):
    if tok.type in (tokenize.COMMENT, tokenize.NL, tokenize.NEWLINE, tokenize.INDENT, tokenize.DEDENT): continue
    if tok.type == tokenize.STRING and tok.string.startswith(('"""', "'''")): continue
    out.append(tok.string)
print(hashlib.md5(' '.join(out).encode()).hexdigest())

In [ ]:
import zipfile, re
for name in ('Thesis_APA.docx', 'Thesis_IEEE.docx'):
    x = zipfile.ZipFile(f'/content/drive/MyDrive/geoproject/thesis_output/{name}').read('word/document.xml').decode()
    t = re.sub(r'<[^>]+>', '', x)
    print(name, {
        'static word vectors (Section 4.4.4)': 'static word vectors (Section 4.4.4)' in t,
        'old ref 4.4.3 gone': 'static word vectors (Section 4.4.3)' not in t,
        'proposal table reworded': 'literature summary in the research proposal' in t,
        'Appendix A/B/C': all(f'Appendix {k}' in t for k in 'ABC'),
        'Table A.1 caption': 'Table A.1 — The 227-tag vocabulary' in t,
        'no bare .380': not re.search(r'(?<![\w.])\.380\b', t),
        'images': x.count('<pic:pic'),
    })

In [ ]:
from PIL import Image; import glob, os
G = '/content/drive/MyDrive/geoproject'
for p in sorted(glob.glob(f'{G}/figures/fig_*.png') + glob.glob(f'{G}/osmfinal/figures/fig_*.png')):
    im = Image.open(p); print(f"{os.path.basename(p):42s} {im.size[0]}x{im.size[1]}px  dpi={im.info.get('dpi')}  pdf={os.path.exists(p[:-4]+'.pdf')}")

In [ ]:
import zipfile, io
from PIL import Image
z = zipfile.ZipFile(f'{G}/thesis_output/Thesis_APA.docx')
media = [n for n in z.namelist() if n.startswith('word/media/')]
for n in media[:30]:
    im = Image.open(io.BytesIO(z.read(n))); print(n, im.size, len(z.read(n)) // 1024, 'KB')
print('settings has doNotAutoCompressPictures:', b'doNotAutoCompressPictures' in z.read('word/settings.xml'))

In [ ]:
# patch 3: stop Word from compressing the 300-dpi figures when the file is saved
p = '/content/build_docx.py'; s = open(p).read()
a = """        if 'w:updateFields' not in x: x = x.replace('</w:settings>', '<w:updateFields w:val="true"/></w:settings>')"""
b = a + """
        if 'doNotAutoCompressPictures' not in x:          # keep the 300-dpi figures when Word saves
            x = x.replace('</w:settings>', '<w:doNotAutoCompressPictures/></w:settings>')"""
assert s.count(a) == 1; open(p, 'w').write(s.replace(a, b))
import tokenize, io, hashlib
src = open(p).read(); out = []
for tok in tokenize.generate_tokens(io.StringIO(src).readline):
    if tok.type in (tokenize.COMMENT, tokenize.NL, tokenize.NEWLINE, tokenize.INDENT, tokenize.DEDENT): continue
    if tok.type == tokenize.STRING and tok.string.startswith(('"""', "'''")): continue
    out.append(tok.string)
print('code hash', hashlib.md5(' '.join(out).encode()).hexdigest())

In [ ]:
# DECK UPDATE -- adds the ablation, FP-Growth and matched-corpus evidence to defence_deck.pptx.
# Every number is read from osmfinal/ablation.json and fpgrowth_check.json. Writes slides/defence_deck_v2.pptx.
import json, subprocess, sys, copy
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'python-pptx'], check=True)
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.shapes import MSO_SHAPE
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
OUT = '/content/drive/MyDrive/geoproject/osmfinal'
AB = json.load(open(f'{OUT}/ablation.json')); FP = json.load(open(f'{OUT}/fpgrowth_check.json'))
A = AB['arms']; C = {(c['comparison'], c['set']): c for c in AB['comparisons']}
prs = Presentation(f'{OUT}/slides/defence_deck.pptx'); BL = prs.slide_layouts[6]
BK, GR, LG, WH = RGBColor(0, 0, 0), RGBColor(0x6E, 0x6E, 0x6E), RGBColor(0xD9, 0xD9, 0xD9), RGBColor(0xFF, 0xFF, 0xFF)
def T(s, t, x, y, w, h, size, bold=False, color=BK, align=PP_ALIGN.LEFT):
    tb = s.shapes.add_textbox(Inches(x), Inches(y), Inches(w), Inches(h)); tf = tb.text_frame; tf.word_wrap = True
    for i, line in enumerate(t.split('\n')):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph(); p.alignment = align; r = p.add_run(); r.text = line
        r.font.size = Pt(size); r.font.bold = bold; r.font.name = 'Helvetica'; r.font.color.rgb = color
def box(s, x, y, w, h, fill, line=BK):
    sh = s.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(x), Inches(y), Inches(w), Inches(h))
    sh.fill.solid(); sh.fill.fore_color.rgb = fill; sh.line.color.rgb = line; sh.line.width = Pt(1.25); sh.shadow.inherit = False
def slide(title, sub=None):
    s = prs.slides.add_slide(BL); T(s, title, 0.75, 0.45, 11.8, 1.0, 30, True)
    if sub: T(s, sub, 0.75, 1.35, 11.8, 0.6, 16, color=GR)
    return s
def table(s, rows, x, y, w, colw, size=13, bold_rows=()):
    sh = s.shapes.add_table(len(rows), len(rows[0]), Inches(x), Inches(y), Inches(w), Inches(0.38 * len(rows))).table
    for j, cw in enumerate(colw): sh.columns[j].width = Inches(cw)
    for i, r in enumerate(rows):
        for j, v in enumerate(r):
            c = sh.cell(i, j); c.text = str(v); c.fill.solid(); c.fill.fore_color.rgb = BK if i == 0 else (LG if i in bold_rows else WH)
            p = c.text_frame.paragraphs[0]; p.alignment = PP_ALIGN.LEFT if j == 0 else PP_ALIGN.CENTER
            for rn in p.runs: rn.font.size = Pt(size); rn.font.name = 'Helvetica'; rn.font.bold = i == 0 or i in bold_rows; rn.font.color.rgb = WH if i == 0 else BK
h3 = lambda k, s: f"{A[k][s]['Hits@3']:.3f}"
def d(lab, s='osm_dev'):
    c = C[(lab, s)]; p = c['p_holm']
    return f"{c['delta_hits3']:+.3f}" + (' *' if p < 0.05 else '')
FINAL, BASE0 = 'final: numberbatch + conf (657)', 'A0 base Alg.2, abstract nouns kept (w2v)'
new = []
# 1 stage-wise build-up (bars on dev)
s = slide('Ablation: where the gain comes from', 'Add one component at a time to the base paper. Hits@3 on OSM dev (6,981); test set never read. * = significant after Holm (60 tests)')
steps = [('Base paper\n(Alg. 2)', BASE0, None), ('+ abstract-noun\nremoval', 'base Alg.2 + abstract removal (w2v)', 'A1 abstract-noun removal'),
         ('+ adjectives in\nthe query', 'A2 + adjectives appended to the query (w2v)', 'A2 adjectives appended to query'),
         ('+ adjective-pattern\ntable', 'A3 + adjective patterns only, conf (w2v)', 'A3 adjective patterns (table)'),
         ('+ adjective–noun\npairs', 'A4 + adjective-noun pairs = 657 patterns (w2v)', 'A4 + pair patterns'),
         ('+ Numberbatch\n= final', FINAL, 'A5 Numberbatch backbone')]
x0, y0, W, H, mx = 0.9, 2.2, 11.6, 3.3, 0.45
for i, (lab, k, cl) in enumerate(steps):
    v = A[k]['osm_dev']['Hits@3']; bw = W / 6 * 0.6; bx = x0 + i * W / 6 + (W / 6 - bw) / 2; bh = H * v / mx
    box(s, bx, y0 + H - bh, bw, bh, BK if i in (3, 5) else (WH if i == 2 else LG))
    T(s, f'{v:.3f}', bx - 0.3, y0 + H - bh - 0.45, bw + 0.6, 0.4, 16, True, align=PP_ALIGN.CENTER)
    T(s, lab, bx - 0.5, y0 + H + 0.08, bw + 1.0, 0.7, 12, color=GR, align=PP_ALIGN.CENTER)
    if cl: T(s, d(cl), bx - 0.3, y0 + H + 0.75, bw + 0.6, 0.35, 13, True, align=PP_ALIGN.CENTER)
T(s, 'The mined adjective-pattern table carries the gain (+0.145). Adjectives added straight to the query hurt. Pairs add little.', 0.9, 6.55, 11.6, 0.5, 15)
new.append(s)
# 2 one change at a time
s = slide('Ablation: change one thing in the final pipeline', 'Hits@3; Δ on OSM dev vs the final configuration; * = significant after Holm')
rows = [['Change', 'OSM dev', '106 set', 'PK n ≥ 3', 'Δ dev'], ['None (final)', h3(FINAL, 'osm_dev'), h3(FINAL, 'eval_106'), h3(FINAL, 'pk_n3'), '—']]
for lab, k, cl in [('Adjective patterns only', 'B1 final, adjective patterns only', 'B1 drop pair patterns'),
                   ('Pair patterns only', 'B2 final, pair patterns only', 'B2 drop adjective patterns'),
                   ('Log-lift weighting', 'B3 final, log-lift instead of confidence', 'B3 log-lift weighting'),
                   ('Frequentist elastic net', 'B4 final, frequentist EN (C=0.5)', 'B4 frequentist EN selection'),
                   ('Bayesian elastic net', 'B5 final, Bayesian EN', 'B5 Bayesian EN selection'),
                   ('No table (nouns only)', 'B6 final, nouns only (no table)', 'B6 drop the table'),
                   ('Adjectives in query, no table', 'B7 final, adjectives appended instead of table', 'B7 naive adjectives, no table'),
                   ('Small parser, not transformer', 'D final, small parser (en_core_web_sm)', 'D small parser')]:
    rows.append([lab, h3(k, 'osm_dev'), h3(k, 'eval_106'), h3(k, 'pk_n3'), d(cl)])
table(s, rows, 0.9, 2.0, 11.5, [4.6, 1.7, 1.7, 1.7, 1.8], bold_rows=(1,))
new.append(s)
# 3 FP-Growth
s = slide('Guided FP-Growth = extended FP-Growth, only faster', 'OSM training transactions, min support 0.0005; identical pattern–tag itemsets at every size, no time cap')
rows = [['Transactions', 'Guided (s)', 'Extended (s)', 'Extended memory (MB)', 'Itemsets', 'Identical']]
for r in FP['runs']:
    rows.append([f"{r['transactions']:,}", f"{r['guided']['seconds']:.2f}", f"{r['extended']['seconds']:,.0f}", f"{r['extended']['peak_mb']:,.0f}",
                 f"{r['guided']['pattern_tag_itemsets']} = {r['extended']['pattern_tag_itemsets']}", 'yes' if r['identical'] else 'NO'])
table(s, rows, 0.9, 2.1, 11.5, [2.2, 1.7, 1.9, 2.5, 1.8, 1.4], size=15)
full = FP['runs'][-1]
T(s, f"Full corpus: {full['guided']['seconds']:.1f} s vs {full['extended']['seconds']:,.0f} s — {full['speedup']:,.0f}× faster, same output.", 0.9, 5.2, 11.5, 0.6, 18, True)
new.append(s)
# 4 matched corpus
s = slide('Register matters more than corpus size', 'Same table type on both sides, same backbone and ranker; only the training corpus differs. Hits@3')
rows = [['Table', 'Set', 'OSM', 'Wikivoyage', 'Δ']]
for lab, ko, kw, cl in [('Confidence', FINAL, 'F Wikivoyage conf (966 patterns)', 'F corpus, conf'), ('Bayesian EN', 'B5 final, Bayesian EN', 'F Wikivoyage Bayesian EN (966)', 'F corpus, Bayesian EN')]:
    for sname, nm in (('osm_dev', 'OSM dev'), ('eval_106', '106 set'), ('pk_n3', 'PK n ≥ 3')):
        rows.append([lab if sname == 'osm_dev' else '', nm, h3(ko, sname), h3(kw, sname), d(cl, sname)])
table(s, rows, 0.9, 2.0, 11.5, [2.4, 2.2, 2.2, 2.4, 2.3], size=15)
T(s, 'Each corpus wins on its own kind of text; on blog-style descriptions the two are level.', 0.9, 5.3, 11.5, 0.6, 18, True)
new.append(s)
# move the four new slides to just after "What each stage contributes"
titles = [next((sh.text_frame.text for sh in sl.shapes if sh.has_text_frame), '') for sl in prs.slides]
anchor = next(i for i, t in enumerate(titles) if t.startswith('What each stage contributes'))
lst = prs.slides._sldIdLst; ids = list(lst)
for k, el in enumerate(ids[-4:]):
    lst.remove(el); lst.insert(anchor + 1 + k, el)
prs.save(f'{OUT}/slides/defence_deck_v2.pptx')
print('slides:', len(prs.slides)); print([next((sh.text_frame.text[:50] for sh in sl.shapes if sh.has_text_frame), '') for sl in prs.slides])

In [ ]:
import subprocess, glob, os
subprocess.run('apt-get -qq install -y libreoffice-impress > /dev/null 2>&1', shell=True)
os.makedirs('/content/render', exist_ok=True)
subprocess.run(['soffice', '--headless', '--convert-to', 'pdf', '--outdir', '/content/render', f'{OUT}/slides/defence_deck_v2.pptx'], capture_output=True)
subprocess.run('pdftoppm -r 30 -png -f 16 -l 20 /content/render/defence_deck_v2.pdf /content/render/s', shell=True)
from PIL import Image
f = sorted(glob.glob('/content/render/s-*.png')); ims = [Image.open(x) for x in f]; w, h = ims[0].size
sheet = Image.new('RGB', (w * 3, h * 2), 'white')
for i, im in enumerate(ims): sheet.paste(im, ((i % 3) * w, (i // 3) * h))
sheet.save(f'{OUT}/slides/_check_new_slides.png'); print(len(f), sheet.size)

In [ ]:
import re, os
W='/content/drive/MyDrive/geoproject/thesis_writeup/'
old='returns the same rules as the extended algorithm about 2,184 times faster.'
new='returns the same itemsets as the extended algorithm up to the full 55,897-transaction OSM corpus, 2,489 times faster.'
for f in ['finalosmdraft.md','finalosmdraft_IEEE.md']:
    p=W+f; t=open(p).read()
    t2=re.sub(re.escape(old).replace(r'\ ',r'\s+'),new,t)
    open(p,'w').write(t2); print(f, '2,184' in t2, t2.count('2,489 times faster'))

from pptx import Presentation
P='/content/drive/MyDrive/geoproject/osmfinal/slides/defence_deck_v2.pptx'
pr=Presentation(P)
rep={'Guided extended FP-Growth over 398,046 weakly labelled training sentences':'Guided extended FP-Growth over 55,897 OSM training descriptions',
 'Guided and extended FP-Growth return the same 153 rules on a 40k sample.':'Guided and extended FP-Growth return identical itemsets at every size up to the full corpus.',
 'Guided runs 2,184× faster and scales to the full corpus,':'Full corpus: 1.0 s vs 2,576 s,',
 'where extended did not finish.':'2,489× faster, same output.'}
n=0
for s in pr.slides:
    for sh in s.shapes:
        if not sh.has_text_frame: continue
        for para in sh.text_frame.paragraphs:
            txt=''.join(r.text for r in para.runs)
            if txt in rep and para.runs:
                para.runs[0].text=rep[txt]
                for r in para.runs[1:]: r.text=''
                n+=1
pr.save(P); print('slide lines replaced',n)

In [ ]:
!pip -q install python-pptx
W='/content/drive/MyDrive/geoproject/thesis_writeup/'
print([l.strip()[:120] for l in open(W+'finalosmdraft.md') if '2,489 times' in l or '2,184' in l])
from pptx import Presentation
P='/content/drive/MyDrive/geoproject/osmfinal/slides/defence_deck_v2.pptx'
pr=Presentation(P)
rep={'Guided extended FP-Growth over 398,046 weakly labelled training sentences':'Guided extended FP-Growth over 55,897 OSM training descriptions',
 'Guided and extended FP-Growth return the same 153 rules on a 40k sample.':'Guided and extended FP-Growth return identical itemsets at every size up to the full corpus.',
 'Guided runs 2,184× faster and scales to the full corpus,':'Full corpus: 1.0 s vs 2,576 s,',
 'where extended did not finish.':'2,489× faster, same output.'}
n=0
for s in pr.slides:
    for sh in s.shapes:
        if not sh.has_text_frame: continue
        for para in sh.text_frame.paragraphs:
            txt=''.join(r.text for r in para.runs).strip()
            if txt in rep and para.runs:
                para.runs[0].text=rep[txt]
                for r in para.runs[1:]: r.text=''
                n+=1
pr.save(P); print('slide lines replaced',n)

In [ ]:
import zipfile,re
for f in ['Thesis_APA.docx','Thesis_IEEE.docx']:
    z=zipfile.ZipFile('/content/drive/MyDrive/geoproject/thesis_output/'+f)
    xml=z.read('word/document.xml').decode()
    x=re.sub('<[^>]+>','',re.sub(r'</w:p>','\n',xml))
    chaps=re.findall(r'^Chapter \d+ — .+$|^Mathematical Formulation$|^(?:Introduction|Literature Review|Methodology|Results and Evaluation|Discussion and Limitations|Conclusions and Recommendations)$',x,re.M)
    print(f,'| images',sum(n.startswith('word/media/') for n in z.namelist()),'| eq-like 3-col tables with (3.',len(re.findall(r'\(3\.\d+\)',x)),
          '| Scope:', 'Scope of the thesis' in x, '| leftover <<:', '<<' in x, '| chapters', chaps[:10])

In [ ]:
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN
D='/content/drive/MyDrive/geoproject/osmfinal/slides/'
prs=Presentation(D+'defence_deck_v2.pptx'); BL=prs.slide_layouts[6]
BK,GR,LG,WH=RGBColor(0,0,0),RGBColor(0x6E,0x6E,0x6E),RGBColor(0xD9,0xD9,0xD9),RGBColor(0xFF,0xFF,0xFF)
def T(s,t,x,y,w,h,size,bold=False,color=BK):
    tf=s.shapes.add_textbox(Inches(x),Inches(y),Inches(w),Inches(h)).text_frame; tf.word_wrap=True
    for i,line in enumerate(t.split('\n')):
        p=tf.paragraphs[0] if i==0 else tf.add_paragraph(); r=p.add_run(); r.text=line; p.space_after=Pt(6)
        r.font.size=Pt(size); r.font.bold=bold; r.font.name='Helvetica'; r.font.color.rgb=color
def slide(title,sub=None):
    s=prs.slides.add_slide(BL); T(s,title,0.75,0.45,11.8,1.0,30,True)
    if sub: T(s,sub,0.75,1.35,11.8,0.6,16,color=GR)
    return s
def table(s,rows,x,y,w,colw,size=14):
    sh=s.shapes.add_table(len(rows),len(rows[0]),Inches(x),Inches(y),Inches(w),Inches(0.45*len(rows))).table
    for j,cw in enumerate(colw): sh.columns[j].width=Inches(cw)
    for i,r in enumerate(rows):
        for j,v in enumerate(r):
            c=sh.cell(i,j); c.text=v; c.fill.solid(); c.fill.fore_color.rgb=BK if i==0 else WH
            for p in c.text_frame.paragraphs:
                for rn in p.runs: rn.font.size=Pt(size); rn.font.name='Helvetica'; rn.font.bold=i==0; rn.font.color.rgb=WH if i==0 else BK
def bullets(s,items,y=2.1,size=20): T(s,'\n'.join('•  '+i for i in items),0.9,y,11.5,4.8,size)
n0=len(prs.slides)

s=slide('Background: people describe places, maps need tags','Volunteered geographic information (VGI) and OpenStreetMap (OSM)')
bullets(s,['Millions of OSM volunteers describe places in their own words, with local knowledge.',
 'OSM stores places as structured key–value tags: amenity=cafe, historic=fort.',
 'People write informally, in paraphrase, dialect and mixed language:\n    “a quiet spot in F-7 with good cold coffee and free wifi” names neither the place nor its type.',
 'Correct tags matter for navigation, urban planning, disaster response and location-based services.',
 'The gap is widest where phrasing is most local: Pakistani descriptions mix English, Urdu and Roman Urdu.'])
A1=s
s=slide('Problem statement','From the approved research proposal')
T(s,'Propose a robust framework that integrates Extended FP-Growth for mining adjective–noun patterns from OSM data, '
  'the Bayesian Elastic Net for feature optimisation, and transformer-based NLP, to classify informal, ambiguous and '
  'paraphrased volunteered place descriptions into their OSM tags; tailored to Pakistan and extensible to global OSM data.',0.9,2.2,11.5,3,22)
A2=s
s=slide('Literature review','Four lines of work the thesis draws on')
table(s,[['Area','Key works','What is missing'],
 ['Interpreting place descriptions','Yousaf et al. 2023; Wolter & Yousaf 2017; Yousaf & Wolter 2018; Tempelmeier et al. 2021','Similarity on nouns only; adjectives ignored'],
 ['Frequent-pattern mining','Akbar 2008; Jang et al. 2021; Lian et al. 2018; Zhao et al. 2021','Used for OSM tags, not adjective–noun structure in free text'],
 ['Elastic net / Bayesian elastic net','Hughey & Butte 2015; Liu et al. 2023; Chen et al. 2009; Yu et al. 2025','Never applied to geospatial language'],
 ['Transformers, weak supervision','Sanh et al. 2019; Silva & Akabane 2022; Xiao 2024; Liu 2024','Weak labels teach the cue; post-hoc XAI explains a model of the model']],
 0.75,2.0,11.8,[3.0,5.2,3.6],14)
A3=s
s=slide('Research gap','Identified in the review (Chapter 2)')
bullets(s,['Methods use either embedding similarity or pattern mining; interpretable adjective–noun patterns are rarely combined with contextual models.',
 'Feature selection, the Bayesian elastic net in particular, is little used in geospatial NLP.',
 'No framework jointly uses pattern mining, feature selection and transformer NLP to map informal descriptions to OSM tags.',
 'Few studies address explainability and traceability in automated OSM tagging, though volunteers must check the result.'])
A4=s
s=slide('Research questions and objectives')
T(s,'RQ1  Does integrating guided extended FP-Growth, the Bayesian elastic net and transformer NLP improve tagging accuracy over previous approaches?\n'
  'RQ2  How much do adjective–noun pattern mining and feature selection add to interpretability and robustness?\n'
  'RQ3  Can the framework explain its results to end users without additional XAI techniques?',0.9,1.6,11.5,2.6,19)
T(s,'Objectives: build the integrated model · analyse what patterns add to contextual representations · evaluate interpretability and robustness · assess explanations without XAI',0.9,5.2,11.5,1.2,17,color=GR)
A5=s
s=slide('Data','Training corpora and evaluation sets')
table(s,[['Set','Size','Role'],
 ['Wikivoyage sentences, weak labels','398,046','Preliminary study: training'],
 ['OSM descriptions, real tags, type words masked','69,895 (55,897 / 6,981 / 7,017)','Final run: train / dev / test'],
 ['Base paper evaluation set','106 sentences','Reproduction and comparison'],
 ['New Pakistani set','199 annotated (157 with n ≥ 3)','Regional evaluation'],
 ['OSM tag vocabulary','227 tags','Every tag ranked for every description']],
 0.75,2.0,11.8,[5.0,3.6,3.2],15)
A6=s
s=slide('Departure from the proposal: the transformer changed role','Built and measured as proposed, then replaced where it lost')
bullets(s,['Proposed: fuse the selected patterns with DistilBERT embeddings and train a transformer classifier.',
 'DistilBERT fine-tuned on weak labels: 0.970 on its own labels, Hits@3 0.142 on the 106 paraphrased sentences.',
 'It learned the type nouns that paraphrases leave out. Sentence encoders as backbones ranked below static word vectors.',
 'Kept: the transformer as the parser (en_core_web_trf). Replaced: the classifier, by a ranker over all 227 tags.',
 'Every component named in the proposal is present and measured.'])
A7=s
s=slide('Limitations')
bullets(s,['Target-register sets are small (106; 157): gains there have the right direction but do not survive Holm correction.',
 'The significant result is on OSM test, the training corpus’s own register (in-domain).',
 'Few Pakistani rows in OSM (209); the regional claim rests on the 157-sentence set.',
 'English only in training; Urdu goes through translation, which costs 5–6 points of Hits@3.',
 'Minimum support fixed at 0.0005; no learning curve for the Bayesian elastic net.'])
A8=s
s=slide('Future work')
bullets(s,['Training text in the target register, labelled from linked OSM elements.',
 'A larger Pakistani evaluation set with independent annotators.',
 'Native Urdu input instead of the translation front end.',
 'Explanation without the accuracy cost: uncertainty on the confidence table itself.',
 'Tune minimum support; fit a Bayesian elastic-net learning curve.',
 'Retrieval test on OSM near a known location, and a user study of the explanations.'])
A9=s
s=slide('Thank you'); T(s,'Questions?',0.75,3.0,11.8,1.2,40,True)
A10=s

lst=prs.slides._sldIdLst; ids=list(lst)
def move(slide_obj, pos):
    el=[e for e in lst if e.get('{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id')==prs.part.relate_to(slide_obj.part,'http://schemas.openxmlformats.org/officeDocument/2006/relationships/slide')][0]
    lst.remove(el); lst.insert(pos,el)
def idx(title):
    for i,s in enumerate(prs.slides):
        if any(sh.has_text_frame and sh.text_frame.text.startswith(title) for sh in s.shapes): return i
# after title slide: background, problem, literature, gap, RQs
for k,sl in enumerate([A1,A2,A3,A4,A5]): move(sl,1+k)
# data before the pipeline slide; departure before 'Earlier work'
move(A6, idx('One pipeline'))
move(A7, idx('Earlier work'))
for sl in (A8,A9,A10): move(sl,len(lst)-1); lst.remove(lst[-1]) if False else None
# put limitations/future/thanks at the very end in order
for sl in (A8,A9,A10):
    move(sl,len(list(lst))-1); el=list(lst)[-1]
for sl in (A8,A9,A10):
    r=prs.part.relate_to(sl.part,'http://schemas.openxmlformats.org/officeDocument/2006/relationships/slide')
    el=[e for e in lst if e.get('{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id')==r][0]; lst.remove(el); lst.append(el)
prs.save(D+'defence_deck_v3.pptx')
for i,s in enumerate(prs.slides,1):
    t=next((sh.text_frame.text.split('\n')[0] for sh in s.shapes if sh.has_text_frame and sh.text_frame.text.strip()),'')
    print(i,t[:70])

In [ ]:
!which soffice || (apt-get -qq update && apt-get -qq install -y libreoffice-impress >/dev/null)
!cd /tmp && soffice --headless --convert-to pdf /content/drive/MyDrive/geoproject/osmfinal/slides/defence_deck_v3.pptx >/dev/null 2>&1; pip -q install pymupdf
import fitz
d=fitz.open('/tmp/defence_deck_v3.pdf'); print(d.page_count)
for i in [1,2,3,4,5,7,14,30,31]:
    d[i].get_pixmap(dpi=50).save(f'/content/drive/MyDrive/geoproject/osmfinal/slides/chk_{i+1}.png')

In [ ]:
from PIL import Image
P='/content/drive/MyDrive/geoproject/osmfinal/slides/'
ims=[Image.open(P+f'chk_{i}.png') for i in [2,3,4,5,6,8,15,31,32]]
w,h=ims[0].size; sheet=Image.new('RGB',(w*3,h*3),'white')
for k,im in enumerate(ims): sheet.paste(im,((k%3)*w,(k//3)*h))
sheet.save(P+'chk_sheet.png'); import os
for i in [2,3,4,5,6,8,15,31,32]: os.remove(P+f'chk_{i}.png')
print(sheet.size)

In [ ]:
import pymupdf
d=pymupdf.open('/tmp/defence_deck_v3.pdf')
for i in [1,2,3,4,5,7,14,30,31]:
    pg=d[i]; H=pg.rect.height; bl=pg.get_text('blocks')
    print(i+1, 'max bottom %.0f / %.0f'%(max(b[3] for b in bl),H), 'blocks',len(bl))
    # overlaps between blocks
    for a in range(len(bl)):
        for b in range(a+1,len(bl)):
            A,B=bl[a],bl[b]
            if A[0]<B[2]-2 and B[0]<A[2]-2 and A[1]<B[3]-2 and B[1]<A[3]-2: print('   overlap:',A[4][:40].replace('\n',' '),'|',B[4][:40].replace('\n',' '))

In [ ]:
from pptx import Presentation
from collections import Counter
prs=Presentation('/content/drive/MyDrive/geoproject/osmfinal/slides/defence_deck_v3.pptx')
print(prs.slide_width/914400, prs.slide_height/914400)
for i,s in enumerate(prs.slides,1):
    fonts=Counter(); sizes=Counter(); lefts=[]
    for sh in s.shapes:
        lefts.append(round(sh.left/914400,2) if sh.left is not None else None)
        if sh.has_text_frame:
            for p in sh.text_frame.paragraphs:
                for r in p.runs: fonts[r.font.name]+=1; sizes[r.font.size.pt if r.font.size else None]+=1
    tt=[sh for sh in s.shapes if sh.has_text_frame and sh.text_frame.text.strip()][0]
    print(i, dict(fonts), sorted(sizes.items(), key=lambda x: str(x[0]))[:6], 'title@',round(tt.left/914400,2),round(tt.top/914400,2), 'minleft',min(l for l in lefts if l is not None))

In [ ]:
import pymupdf
from PIL import Image
d=pymupdf.open('/tmp/defence_deck_v3.pdf')
def sheet(pages,name,dpi=40):
    ims=[Image.frombytes('RGB',(p.width,p.height),p.samples) for p in (d[i-1].get_pixmap(dpi=dpi) for i in pages)]
    w,h=ims[0].size; c=2; r=(len(ims)+1)//2; S=Image.new('RGB',(w*c,h*r),'white')
    for k,im in enumerate(ims): S.paste(im,((k%c)*w,(k//c)*h))
    S.save('/content/drive/MyDrive/geoproject/osmfinal/slides/'+name,quality=55); print(name,S.size)
sheet([2,4,7,9,11,16],'v1.jpg'); sheet([20,22,23,28,29,31],'v2.jpg')

In [ ]:
from IPython.display import Image as I, display
display(I('/tmp/s.jpg'))

In [ ]:
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.oxml.ns import qn
from lxml import etree
P='/content/drive/MyDrive/geoproject/osmfinal/slides/defence_deck_v3.pptx'
prs=Presentation(P)
def title_of(s): return next((sh for sh in s.shapes if sh.has_text_frame and abs(sh.top-Inches(0.45))<Inches(0.05)),None)
def ttext(s): t=title_of(s); return t.text_frame.text if t else ''

SHORT={"Earlier work: the gain on the base paper's own set and on a new Pakistani set":"Earlier work: gains on the 106 set and the Pakistani set",
 "Why one more run: the earlier evidence rested on hope in four places":"Why one more run: four weak points in the earlier evidence",
 "The base paper reads the nouns and throws the adjectives away":"The base paper reads nouns and drops the adjectives"}
for s in prs.slides:
    for sh in s.shapes:
        if sh.has_text_frame:
            for p in sh.text_frame.paragraphs:
                for r in p.runs: r.font.name='Arial'
        if sh.has_table:
            for row in sh.table.rows:
                for c in row.cells:
                    for p in c.text_frame.paragraphs:
                        for r in p.runs: r.font.name='Arial'
    t=title_of(s)
    if not t: continue
    tt=t.text_frame.text
    if tt in SHORT:
        r=t.text_frame.paragraphs[0].runs; r[0].text=SHORT[tt]; [setattr(x,'text','') for x in r[1:]]; tt=SHORT[tt]
    if len(tt)>50:
        for r in t.text_frame.paragraphs[0].runs: r.font.size=Pt(26)

def bulletize(tf,size):
    for p in tf.paragraphs:
        if not p.runs: continue
        full=''.join(r.text for r in p.runs)
        pPr=p._p.get_or_add_pPr()
        for e in pPr.findall(qn('a:buChar'))+pPr.findall(qn('a:buNone')): pPr.remove(e)
        if full.startswith('•  '):
            p.runs[0].text=p.runs[0].text[3:]
            pPr.set('marL',str(Inches(0.35))); pPr.set('indent',str(-Inches(0.35)))
            etree.SubElement(pPr,qn('a:buChar'),char='•')
        elif full.startswith('    '):
            p.runs[0].text=p.runs[0].text.strip(); pPr.set('marL',str(Inches(0.35))); pPr.set('indent','0')
            etree.SubElement(pPr,qn('a:buNone')); [setattr(r.font,'italic',True) for r in p.runs]
        p.space_after=Pt(14)
        for r in p.runs: r.font.size=Pt(size)
for s in prs.slides:
    for sh in s.shapes:
        if sh.has_text_frame and sh.text_frame.text.startswith('•  '): bulletize(sh.text_frame,22); sh.top=Inches(2.2)

for s in prs.slides:
    tt=ttext(s)
    if tt in ('Literature review','Data'):
        for sh in s.shapes:
            if sh.has_table:
                for row in sh.table.rows:
                    row.height=Inches(0.8 if tt=='Literature review' else 0.6)
                    for c in row.cells:
                        c.margin_top=c.margin_bottom=Inches(0.08)
                        for p in c.text_frame.paragraphs:
                            for r in p.runs: r.font.size=Pt(17)

fixed=0
for s in prs.slides:
    for sh in s.shapes:
        if not sh.has_text_frame: continue
        ps=sh.text_frame.paragraphs
        for i,p in enumerate(ps):
            if ''.join(r.text for r in p.runs).strip()=='Full corpus: 1.0 s vs 2,576 s,':
                p.runs[0].text='Full corpus: 1.0 s vs 2,576 s, 2,489× faster, same output.'
                [setattr(r,'text','') for r in p.runs[1:]]
                nxt=ps[i+1]._p; nxt.getparent().remove(nxt); fixed+=1; break

s=[s for s in prs.slides if ttext(s).startswith('One pipeline')][0]
boxes=[sh for sh in s.shapes if sh.has_text_frame and sh.text_frame.text.strip()[:1].isdigit() and sh.width<Inches(2.5)]
bot=max(b.top+b.height for b in boxes); moved=0
for sh in s.shapes:
    if sh.has_text_frame and sh not in boxes and bot-Inches(0.3)<sh.top<bot+Inches(0.3): sh.top=bot+Inches(0.2); moved+=1
prs.save(P); print('ok stage2 fixed',fixed,'boxes',len(boxes),'moved',moved)

In [ ]:
NOTES={
'Geospatial Tagging':"""Intro: name, programme, supervisors. One line: "I extend a published OSM tagging method so it also reads adjectives."
Title keeps "Extended FP-Growth": the guided variant I use returns the SAME patterns as extended FP-Growth, just faster (slide 26).""",
'Background':"""VGI = volunteered geographic information (maps made by volunteers, e.g. OSM).
OSM tag = key=value label, e.g. amenity=cafe. Task: description text -> correct tag.
The F-7 example names neither "cafe" nor the place: the type must be inferred.
If asked "why does it matter": search, navigation, planning, disaster response all query tags.""",
'Problem statement':"""Read it as written in the approved proposal.
Examiner may ask "did you do all of it?": yes, every component is built and measured; only the transformer's role changed (slide 15).""",
'Literature review':"""Proposal reviewed 26 works; these are the 4 groups.
Key message: each line of work lacks one piece; nobody combines adjective patterns + Bayesian EN for OSM tagging.
Liu et al. 2023: LASSO/EN over-select -> why I don't read every non-zero weight as meaningful (I use the credible interval instead).""",
'Research gap':"""Four gaps from Ch1.4. Gap 4 (explainability) is what RQ3 answers.""",
'Research questions':"""RQ1 accuracy, RQ2 interpretability/robustness, RQ3 explanations without XAI.
Honest answers (slide 29): RQ1 yes, and the gain comes from the mined adjective patterns, not from the elastic net.
XAI = post-hoc explainers like LIME/SHAP; my explanation is the model's own weights + intervals.""",
'The base paper reads':"""Base paper = Yousaf, Schwartz & Wolter 2023, KI 37(1):69-81.
Their footnote 6: attribute adjectives are ignored. That's the gap I exploit.
Example: nouns place/bread; adjectives parsian/best add information.""",
'Data':"""Two training corpora: Wikivoyage (preliminary, weak labels guessed from type words) and OSM (final, real tags).
OSM: 69,895 English descriptions after masking + dedup, split train 55,897 / dev 6,981 / test 7,017.
Masking = the tag's own words removed, so the model can't just read "cafe".
106 set = base paper's own test sentences. Pakistani set: 199 annotated; 157 have n>=3 = at least 3 descriptions (primary subset).
227 tags = the lexicon; every tag is ranked for every description.""",
'One pipeline':"""Five stages. Numbers per stage on the next slides.
lambda = blend weight: score = (1-lambda)*noun similarity + lambda*pattern score. 0.50 with Numberbatch, chosen on dev.""",
'Stage 1':"""en_core_web_trf = spaCy's transformer parser. Finds nouns and the adjectives attached to them.
95%: share of the 106 base-paper sentences that contain at least one adjective the base method discards.
Ablation: small parser instead costs 0.011 Hits@3 (significant).""",
'Stage 2':"""FP-Growth = frequent-pattern mining (finds item sets that co-occur often).
Guided = only mines itemsets containing an adjective/pair AND a tag; extended = full tree.
Min support 0.0005 (fixed, not tuned: a stated limitation). Result: 657 patterns = 496 adjectives + 161 adjective-noun pairs.
Identical output checked at 5k,10k,20k,40k and full 55,897; 2,489x faster on full.""",
'Stage 3':"""Bayesian elastic net: L1+L2 regularised regression with priors, sampled with NUTS (a Hamiltonian Monte Carlo sampler).
0 divergences = sampler healthy. R-hat <= 1.012 = chains agree (want ~1.00, <1.05 fine).
Median ESS 1409 = effective independent samples; enough for stable intervals.
2.3% credibly non-zero = 94% interval excludes zero -> strong selection.
Jaccard 0.69 = overlap of selected patterns with the ordinary (frequentist) elastic net: they mostly agree.
If asked "why 94%": ArviZ default HDI; any fixed level works.""",
'Stage 4':"""Hybrid score. Noun similarity = base paper's idea (word vectors). Pattern table = my addition.
Final table = unselected confidence table (conf = P(tag | pattern)). EN was not better for accuracy (slide 23).
lambda chosen on dev in the final run; earlier work used cross-validation on the test sets (admitted on slide 17).""",
'Stage 5':"""Hits@3 = share of descriptions whose correct tag is in the top 3. nDCG@20 = rank-weighted score over top 20.
Worst-case ties: if tied, gold is ranked LAST among ties (pessimistic, so no lucky wins).
0.462 = my reproduction of the base paper's best (clustered, Alg. 3) on the 106 set; mean |diff| 0.020 from their published numbers = faithful reproduction.""",
'Departure from the proposal':"""DistilBERT 0.970 = accuracy on its own weak labels (easy: labels came from type words in the text).
0.142 Hits@3 on paraphrased 106 set = it learned the type words, which paraphrases don't contain.
Decision: transformer kept as the parser; ranking done by similarity + patterns. Not re-run under leak-free labels (stated in 5.5).""",
'Earlier work':"""Preliminary Wikivoyage study. 106 set: 0.462 (base, clustered) -> 0.509 ours. Pakistani n>=3: 0.510 -> 0.618.
Weakness: lambda chosen using these sets -> next slide explains why I ran a final pre-registered run.""",
'Why one more run':"""Honest self-critique: tuning on test, tiny test sets, weak labels, selection unproven.
The final run fixes all four: separate dev set, 7,017-row test, real OSM labels, ablation of selection.""",
'The final run':"""Pre-registered = protocol written before seeing data. Harvest 2,875,608 OSM descriptions -> 69,895 usable (English, masked, dedup).
Split by element AND place name (no same place in train and test). Test IDs hashed at split time; test scored exactly once.
Adoption rule: change kept only if dev 95% CI excludes zero. Holm = correction for many comparisons (20 in final run).""",
'What changed':"""Side-by-side of preliminary vs final. Key: the final run's choices never touched test.
Learning curves: 10-100% of train, to check overfitting (slide 28).""",
'On 7,017 held-out':"""Main result. 0.189 (base paper method, Alg. 2) -> 0.380 ours. Delta +0.191, 95% CI [0.180, 0.203], paired bootstrap; Holm p<0.0001.
"In-domain" = test is the same kind of text as training (OSM descriptions): say this before they do.
Q: why Alg. 2 not Alg. 3? Post-hoc I ran Alg. 3 on OSM test: 0.102, lower than Alg. 2 (0.189). So Alg. 2 was the stronger comparator.""",
'Every configuration':"""All six pre-registered configurations on test; lambda fixed on dev beforehand. Numberbatch + confidence table is the best and the adopted one.""",
'Same direction':"""Same final config on all 3 sets, nothing tuned on them.
106 set base here = 0.434 (Alg. 2) vs 0.462 on slide 14 (Alg. 3 clustered): different base variants, both reported.
Small sets not significant after Holm (p 0.85 and 0.23): too few sentences (106, 157). Direction consistent.""",
'What each stage contributes':"""Paired bootstrap deltas on dev.
+0.145 = adding the mined adjective table (THE gain). Numberbatch over Word2Vec +0.031.
Elastic-net selection -0.023, Bayesian EN -0.037 vs unselected table: selection costs accuracy.
So Bayesian EN is kept for RQ3: every weight has a credible interval = built-in explanation.""",
'Ablation: where the gain':"""Add one piece at a time (dev, Holm over 60 tests, * = significant).
Base 0.199 -> abstract-noun removal ~0 -> adjectives straight into the query HURT (-0.016) -> adjective table +0.145 -> pairs +0.006 (not sig.) -> Numberbatch +0.031 = 0.381.
Story: adjectives help only when mined as patterns, not when thrown into the similarity query.""",
'Ablation: change one':"""Remove/replace one thing from the final config. Biggest drops: no table (-0.145), naive adjectives (-0.159), pairs only (-0.086).
Note: frequentist EN is higher on the 106/PK sets (0.585/0.656) but lower on dev; rule says decide on dev. Not significant on small sets.""",
'Guided FP-Growth':"""Guided vs extended, same data, no time cap. Identical itemsets every size (470,451,398,357,353: fewer as n grows because min support is relative).
Full: 1.0 s vs 2,576 s, 2,489x; extended uses 1,375 MB.""",
'Register matters':"""Register = style of text. Same table type, only the training corpus changes.
OSM table wins on OSM dev (+0.131); on blog-style 106/PK the two are level (not sig.).
Lesson: train on text like the text you'll read (future work #1).""",
'It holds up':"""Train-dev gap 0.020 at 100% of data = not overfitting.
Near-duplicates: 930 test rows similar to training (Jaccard>=0.6) removed -> still 0.366 vs 0.200.
Synonym leak: 305 rows still contain a synonym of the tag -> removed -> 0.370 vs 0.182. Gain survives both (post-hoc).""",
'Answers to the research':"""RQ1: yes, 0.189->0.380 in-domain, carried by adjective patterns.
RQ2: readable weights, robust across backbones/audits; selection adds sparsity, not accuracy.
RQ3: yes, credible intervals from a converged posterior; no LIME/SHAP needed.""",
'What the thesis shows':"""Four take-aways. End with the limits yourself (confidence, honesty).""",
'Limitations':"""Say them before the examiner does. Pakistani OSM rows: only 209 survived filtering (26 in test).
Urdu via translation costs 5-6 Hits@3 points (Section 4.5).""",
'Future work':"""Top priority: target-register training data + larger Pakistani set with independent annotators.""",
'Thank you':"""Have the web demo open: https://bat.tail3da3d0.ts.net (try sentences beforehand)."""}
from pptx import Presentation
P='/content/drive/MyDrive/geoproject/osmfinal/slides/defence_deck_v3.pptx'; prs=Presentation(P)
miss=[]
for i,s in enumerate(prs.slides,1):
    t=next((sh.text_frame.text for sh in s.shapes if sh.has_text_frame and sh.text_frame.text.strip()),'')
    k=next((k for k in NOTES if t.startswith(k)),None)
    if k: s.notes_slide.notes_text_frame.text=NOTES[k]
    else: miss.append((i,t[:40]))
prs.save(P); print('no notes:',miss)

In [ ]:
NOTES['Guided FP-Growth']=NOTES['Guided FP-Growth'].replace(' (470,451,398,357,353: fewer as n grows because min support is relative)',' (470, 451, 398, 357, 353)')
NOTES['Every configuration']="All six pre-registered configurations on test; lambda fixed on dev beforehand. The adopted one (Numberbatch + confidence table) was chosen on dev, before test was scored."
prs=Presentation(P)
for s in prs.slides:
    t=next((sh.text_frame.text for sh in s.shapes if sh.has_text_frame and sh.text_frame.text.strip()),'')
    k=next((k for k in NOTES if t.startswith(k)),None)
    if k: s.notes_slide.notes_text_frame.text=NOTES[k]
prs.save(P); print('saved')

In [ ]:
!cd /tmp && rm -f defence_deck_v3.pdf && soffice --headless --convert-to pdf /content/drive/MyDrive/geoproject/osmfinal/slides/defence_deck_v3.pptx >/dev/null 2>&1
import pymupdf, io, base64
from PIL import Image
d=pymupdf.open('/tmp/defence_deck_v3.pdf')
ims=[Image.frombytes('RGB',(p.width,p.height),p.samples) for p in (d[i-1].get_pixmap(dpi=36) for i in [2,4,7,8,9,11,15,16,17])]
w,h=ims[0].size; S=Image.new('RGB',(w*3,h*3),'white')
for k,im in enumerate(ims): S.paste(im,((k%3)*w,(k//3)*h))
S.save('/tmp/s.jpg',quality=50); print(S.size)

In [ ]:
NOTES['On 7,017 held-out']=NOTES['On 7,017 held-out'].split('Q: why Alg. 2')[0]+"""Q: why Alg. 2 here? Alg. 3 (clustered) IS compared: reproduced 0.462 on the 106 set and beaten in the preliminary study (0.462 -> 0.509). On OSM test it was run post-hoc: 0.102, below Alg. 2 (0.189). So Alg. 2 is the stronger, harder baseline on OSM."""
prs=Presentation(P)
for s in prs.slides:
    t=next((sh.text_frame.text for sh in s.shapes if sh.has_text_frame and sh.text_frame.text.strip()),'')
    if t.startswith('On 7,017 held-out'): s.notes_slide.notes_text_frame.text=NOTES['On 7,017 held-out']; print('updated')
prs.save(P)

In [ ]:
import os, time, hashlib
W='/content/drive/MyDrive/geoproject/thesis_writeup/'
for f in ['finalosmdraft.md','finalosmdraft_IEEE.md']:
    t=open(W+f).read(); print(f, len(t.encode()), time.ctime(os.path.getmtime(W+f)))
    print([l for l in t.split('\n') if 'comparisons, Holm' in l or 'comparisons on each of four' in l])
print(time.ctime(os.path.getmtime('/content/drive/MyDrive/geoproject/thesis_output/Thesis_APA.docx')))

In [ ]:
import base64,gzip,subprocess,hashlib,os,shutil
P={"tools/renumber.py": "H4sIAAAAAAAC/7Va3XLbRpa+x1P00lsLQAYhgqREiRp7LHvksRLLcSxlXVmSTjWBJtkW/gKAkjiSqlI18waZR8gTzOXuZWpuM++QJ9nvdAMgKFsTJ1PjKtv46e7zf75zDthqtY7jXGQFe7bgaSEy1mPWCS8WIuKF9HnInidZtAxxk8Q243HAToVPN8xzd5l16iepsB2WiXgZTbF9umIRTx2Wi3DW9hfCP3cNY5nzuRiydFUskrjHiiQJ8+1qi5uu2EzGPEzyKMj4rHCjgPGUM8Zu8Nd13c3X3xwfHR3RGimEMI4uRLYC+ZnAeb5gMsfNZSaLQsSsWGTJcr5gSSxYEgbtx7G4JPYO8EYwxR3jOYmfM8H9RWNrfaKRFzIMWZrIGKt4wThbCB7IeL7tQ2OkiUtZLNSJOY8EK2QR4l9xhaU5m4pZkgnXaLVahozSBJrOBNSzyg0jhZ5xWayw/hE9cnk2vxh5E2d9050YeebjNfQcW7TDdjPQt2zj87d4nJlWabn89zelaXB1xqehwP/P5XyZ0cXRt0uu39mm8eqrE7V15LX3J+Ngy/r9cOyOg4f2dcfp3prGy+PTM1owM6+x9JZe57TIuYED3CTZzc/ffX/TvikSG8/VEnvLNN4cPadNwvWTKJWhsLB/PL3+/O2tNc4f2tY1HXtrm7ZhGIGYsQICiiK3CntowNAMKjqXcLD2Y1Y6E67W2iQN0rIARK7NUmhzyK5vHWaWkle3SvzqRutA3d2qE2ARFjFJRnbhWoHEQVZmvntQB4FFymA/f/dXZrkP7f80HVaQj7sn9pAFo5r4ZBTBWhMwFMFQbl5kMoVh7qcxA5Hr667Tu71lllbcxylU8vxqCua78ZalJdZOYCthtHnvlUgRGREJTaq3SSoTxTKLWVAajqz0TTKzLpMswPbyLd25md5m5rWVEXKwpkV7HBaXpk5BJXbzNJSFZbqmpkJejiCz0lFnop/IGR4+fsR6yB14SEFSZJbPHjLPZp/05wGiXEcH67V3yaH67YE6W4RSS8IePVr7j8pwPj3y1GVIQWezx+WtZs+b2MTVrpZFyYNnJXfrJZ/M5QPKpW3P3SP+PHeAy/0ytYIipScRbBgCKnPfIyGBNWiZtrOUZ+cqByHnCZXtmKhCniUz9SoSyL9BEibzVaUW5KcF6SANuS8W2EWasrAYaTy5wGkJYqJnGwVJl/lGvpzmuBzB0d5S1gNZ5Dyl35/+5jBZsEjGMpK5yNnYqpIO641db2yP43H8u98dfalvb0bvHk+2Hj+mp5cLJFv281++p/ytcnOW5HlbxEWWAB6Q+3MEslPrm5nHTUq0I5m+JxuCZ0j75MnRlz1398kTh+mjTXu9Gbz7iZjNpC9BgC24PiAH2ol2JK+gYsHSTCK+NkXoborQ3RBhGUN3oCb9xQajn0qq5npAXDdP22T9DPsVHuIUIAvtU9BV8Dkpbw4NxITCTcZ7wybfvU3Vkxl/+j8YtbPB9y/QITGQXeA6NeNeX+m7cZ49MShLcYdNKVGR8+iYQe6JNVbgWWzRAiQk7eMakOk9gtBhM7Ppy4hCf4Ej46RAAlwSeWA0UjsfDfudCbCr3G6StKZaB8qFil66IZKCZ/4CqlxjIhsHlBFRyJiIYb7aiB8kgVlhUpx13XWhUxYB8MuyDMgVDaHqkXOxonTYqEuqUkTFIU8NEPiGgmoNgoaqR1R0TVS8ae3cA0xgN+TRNOAsGkJH6xXXZcJtlQ9aDqPsbt+Wu6D6Wcjn+SOV/DcIAZ2sCp7sGp/uULqm027XZMrMqch0icw/p/GL+HSH3NYdepEqj0pSH5VIw85MXllRiTbnoF9hltKFeposG3xB1Jru1ZBZ2hYuT1F1BZZ17rArAJBTo5m+t237w0clg73JRsquFHdNrN9egzh8lXyq8qPttRuFVOCtWICy9YDl5zIlr4koqJM4XCEnL30k7TmbJlgVyljkhvqX3KkC1XEMVKXgkw57SSFAfisyXghLrS1VAwyEBlDrq4CA+W8+MM84sKHkl/Ya7fwkLmS8FOqBOm0kCf5QACplQve0QdmdGNFYpckaUNddxzemPNBuTwxDkQmlB/CsrVBzel6FswqeYc0H+9RaoM6JC4RqKAJ0KQRghQhXFY1kg8bofLIWG1xWDgFrnsMnk9shliOvqbxI8pT1fp0mYCbTPtjUmFRFkTrcxRYLfdV/PKrpjZJfINl+fB2Dbuu6seO2BcI+dR+t6ztn37bgCFVOhWg40mlYBbc2uWHPZbrgyHU8K7+uqgRqUrKlr9EKCptnPKXOZRVN0ck56zSXG6fPvnh9BGOiWn+g6pry2LIEwd9covU5q6+hmwsqH+htxuNzcmwsPornocwXujBhgch91JU6PooFurBAGypWHRe2qnUmXq6IWFwkBj3tdgfsC6jvtEBRVJzwlOCrroemPCd5UvSgjBhKZSrISxW+ZRxXBC93Dmiy4hgK7Tiii2ehRPZ9K8/lRbJCuwu5snSZs2UuAgVBRCDD+ShbOCAiL5bBytFVZUEUxQUP4Z6a5kKgXUWSMP4JcVq3KQbk9zq7MGFcKGuo0wk0lTFzuCckf83PZV7wWBrN01z2VRYs1YY3ScTjtrqVcbpUzFUBgxwEzcS5HghQPZzUpipdv2ho0siWdHTD2kECtshyURKIUBMNZAanC6GN+m2Y+FAFGNf2R1pAHs0k0JW0NcuSqEJSxyCe640y9sNlQDuh+ExrubK3uMJpsYb8g3qOMXC7SGJ5oVwQmkSNNltmuM6opaHxBXWeZd2BhKbcesBeJJfqUFoDuyAscELNWIq6Ncl5WNcjBbriJVoDfY6tipsDphIh+nklZfnOYTqGHpa1DsLTX/SqCYCJ62+QrxduFJj1KAAppRwjgEGai5jDukesqq+RZVqHiNxLHjrsOJIw0N//AnRx2H+x00se4cLb3+9RsjdH3f0J6k6GLS84csFroZZ9LXHd7XQ65aLBpFmc1n+w63+SJda/4Hkhhdqyo7f0OtW5LyUWvNQHetWBOzXVZDaDF2LJH0UYcb2qX67avZfsZz/+kF2Q52Hj5+L8xx/CH3+AG2quuyULXkXkCOailWdymi8k3HpDBb3uvWRO/Fci4hmt7g/K1b0162FEbwb7Jbt71ZtnyYJY8fZ3S3l7fbxqpPvSV2DiylvssgrmB0w7Ab2r/QWW3ayZ7xS4Y2t8OXpn2ZOtcXDdvx0TVRxAZe7h60Pmy2Jd4JKL0GStyvlm0+Xr6rKvRggn60byNzm4EqIcMMDPTepDzIbDqzKAGLJMdHGkKrRF+j+vb9rDO8fOTGpAAI9PnpjUMNQwj0dr8MOidT9gUAGOQywzE/NlyDNJaXomuMK4XIRlcqDcohIeWEL5pLMWMtpaHzqJTssuNOAF130A9UpIs3MB0SoDm2UiqtKju+auUh/Y0rozNmXEc8j/rzHbI4aKktOC5+f3c9r8Y0JhUXMkTMf/KhWwj8htnH59ouuEG3bG8Bd95QWy/pQEXDHr49BtsxvjhkEVcwt6umGHHxYIJaSiFSYTU7tKW15hvcMO1b+v9d5nKBNoShtD8UBTHpQDBA2b9e3P331PK4CwMlOlQ6AO/G+HPdfnvOYF9BA3uD/AYvWssf4ff8bSExpZLCNUjKkaBeNtuVJNM2LNa2yl1Ixi/Svda5L6qB6hBRul0KWewegTUu0AELhQx7xd8wYZ6PmlkPMFfI0Ke7UE5Tcbj6FQIvZ8s9vXB1EoNpWrhcm05G9Qr1W2rbVNY8G8aPtUlSD7lyZ7uiHNNEkKarRTpKucR2kIpd/cwdkb9tP/KishaY5R2ZyjEOaM5msia/MYkcwLgrc5/CNCxQM65MC/IR2VNB8yuCQccwNMecpLLEV3c0qj5zpEzBJNzWF96bA37h1wPXMbAHvoMkshjKtcAeYEu4kvdbrKlqSHqSguhYipXFPuIwsR5ZS0QuowVHhRxYe4Po7Z1uss8YVQHWRlCSLADp+dsNPjP5588QcsIxdQJGBfeH01kwDJE9RDc6U+2v0HnL3FrDR1gZcDOE3X2wWri6JI8+H2dpBIN8nm217H9bz+zrY36HR6Oy79N+g2hkemshmppUS8z5TYQD0XAdsciIFmbT/lanESlV7IQxJx6yhY+hXr5N6v8xWMBeRRmehE8Bw5kLYTxG9ZHsK7R5z3d+/hezDY7nS8nrfb7++iJkAx43X6TeZVVUDMqwuHPVUGbNYIb9xSImXIrUNVB2cJOFVmREmsyvLKx7dc9grl99coFB326uuhypy6uHnBw/DjfEKl2/uDvbbX7u/t77f7O32vvd9ksy51iNX6xmGfK3Y3ix/iVhVAyNpL9TUP+DFHQmmTJwVV60HMwyLHb1gh/EUsv10qE5ArnRGscL8cJ6NSiAkE9JbTVU4uqg3QhwH63S5Z4F4TwHV29rp9b8dV/+01xSpLKxKqrrK+VMpGpQVFJoWo+h6VOCiGRJbpAaqCHzmr/LuKJGQ0NF5KaFWXZ6UYGUPP5MN3EAIkqPatiBqNc9RWXnfL6kIcb6cHcbydwf2Wevq80+3u7+zv7zZlQRFMcuA/h71QZqkK4jNtkB1I9KaE9D/xGrwucKtS9BraLyTXLUxIJ/jo5wow/FmyzCgwStHfoPEM2SkVdblG6lPkFlGshmh20DzlyMJW83WjjoOcuwMtb6/jQd5et3OP+Txv+z3qsN1Be39vd88lOVz87fTcK1N/YKOZFk0F8qI5knoAWas5ATI1FKuLVxpjAe+4DNU8Lm/MsB6wQzX9kFfscL2BPgtINe4aCZU16DOJOqb6arU+geZgNHER1auJOiLuYDd93CnPsg+AyQQ75QxQDGmHPsRRlEediRsmlyJrfH5TdAkXXAogkVsoTT/9zwN2LkSKrpTMQVOPLKBRi/7iQ512TtaNRO1Qkr6aoWKxrPeK/HuHXW2O9ippSGLstq7o0xVdCJpUNuWFwOWlqwlaSGxio5tAbrIEiS3RyKYLjlq9FromVH0kqx8ANKHch2xXD/rBMnkDVfmb9tcVv7aQHkXVR5SPN4yPh+QhBgpaofH41RF9/x6pXpVScg3BCK4NBCYWAbwlBLf+ZeB1WvRQIa/bwNkt3Tw6bAM+XXOjhVQdMJIBowaZGAOzVVJo/eZUAI62kOOhhDK6XYp8lz0FTxcJEGZ34ABawZbmro5w3Zp/yKM3AY6wdTdNrIDAJqq0fi2WEJs1mFD5MnMVeLgVm8QQsdnXbNY4otv3D9nsToDOTLfxanKlgLkJ1QTOxx8B56f3gLPDvjo9vIvQ2rAfku9NCJpqmGr9+8CJNLeBTpXGvO6GYWuo0qOJD1nuT0hDZVnW+lXFGPFA1Zhb1WCuKr8+MJ5X+piuwvS4wzV12q2r++aU4t141N0bT5i71fxZwkYmUtvK8n001LcuTcXtic4WZS4pMwlSA2WRYtRcOZzQnLtf/pqo+rbQ+Erecz3X7bndrhr/62RMXxdVI3tQftmrZwuZoE8neRKiYTTEt7kWin6IQXkzU98fG5+2xjfqA2PVnqgNyF4zs+dey1tT6VuqX3PweC4sKLHbsycOLTTu+aVH4/Plmo6iMqwUR5+cVLr+NqdvUp2JQdOdu6x+0g9fjDRTP21Qv1kyq19VDNVEiafVa3P9KWD9YypaRBihVU7Tp5u12quXYNG2jfpXTlh0adounSDoG9H/A5B6GpypJgAA", "ch3_math.md": "H4sIAAAAAAAC/31aXW8cyXV9719RwMLeHnrI5fBDWpHmwpRESWtracWid5EIMl3TXTPTZHdXu6uHwzGHhqSVnVWA5MFeBEGMAAt7EyB5WwGGgShAAC3yFox+g+eX5NxbVd09Wm4eJM5018et+3HuubfmHXFrJItKlWJTLJ58Lj6S1UhlskoimYo7uszGKb7oPAiORokRkRtsKlkpIzBWVHI4TPIh/ppTIfNYKBmNaMBQCT3gIUVSqDTJlUhykbU3GGCDtVqCrSBWJiqTvls5lpXs8qckK1KVqbxiWXgXemxUVWFrs8vf1C/G/NqIkSqVkPjHq6hBkif8IqhGsqrP4FcplRmnlSFhvSTbeJhOhc7XxBHtM836OjW8ZKTTVEWViukwNP2hf5nHwX6/X6qzxImRJqZaC4J33hGbaz3xoNR9HKIlTxDcV5U4En3lFSnOdCT7UHk5tQff2Lguflyo/GFVKlV9JAsaZbxe+9JAubJQ5ZrYF0UqIzouqbBgPcXBSGIwFD/UacwbDMO4IxaffSaO/NFMBaGkMckQIkthIo1TmrASM4GhlRbqTJVTnlzZmay5UlXjMmdDBSwT1BGrqFTSkDfoMoYeISevZ/diD0l4SqkGtI0UpcxPvfd0Ra4rPItSEmcAD+FT0KudgM4LA6dJNaVlpShKFScRj7BrikIb1iu953M6C7k97PKTkaL9rcY16xPzA0hZwqykx1T2FWwce1O7HSS5m7hNixcyKY0I4y5v09nFUwwwBYTDnlBZVUrYmLaMob1UF+S7ATucMhXml5Xp2mBh5eJgeUUnGyRV7VqtcMNH90bbI/kNeCk8TKfe0TbETxRUY+pgYWW1vIIiGcFyjiXs2WyoZrIwywNFTNavRvA8OrrZCYJVcYgDW9eMdF5hFSh1nOMwk6QaYf4gOYeU5Pm8cd/wNnaQCFfYR1e6YsUUuqK/CCm5wpowSZbA8TuBgG9l+kzFu9hvv95PxidkjDOFzVI9UeVqBPePnTNin4SjEiLLtBjBhoAYMYEfGlrnwTfXWTz5HYllrdm1sCVhaooHM6YvRvxcZjreMdhjetynET+Hnu9ick4eKGGSMm8FrfhYhA+dy2yubbod3TgO25hB5A55554I9200/geLR5/+XXxs/SImpyy9JXGu/tSGfJLTPmfYRJfi3EfzxXpX9C5/Nvt4Zg1xfnxit+jB0jAAwAyOn9cin5BLwctIkLVgv37OAKmjaAz/NuSbAOkqyRQPXvYOck+ozzplpLz/bYoHbq2MPTQIkEUwtfbY9hrOlXMj6zB789wKvqQa/nABP92hgLt00DUuCl1aP8sDMhsF6IceDcyINE1I6Xdu7WTsScmH8RI2hy2GpZzItCs+zBJkq/95bk6TrviueDiRGT70btzY7CAEvv/9g78iPJ/R9uGHJOrsAuYiK9wWO9h/8Xe/cae4nIn3xOz27IMPgmDfehaJSNFcIlvBsNYqfqnFi38Vbz61pyPtZeOsPuWbT7HT+tr6+vq2hYNzixMyPpOkfpoUXOGRXhuKFVXvSyCdNq7ZhefuiQtR4AQszUVx6eURl7seSZG5ZBqoM5naVEuKGyG1GHFt+3prsa0b11rxyg7du9b7ltCD56ys7BujI5s4kQ+riVqKMFpAElSvrawISpp5WCC2Oj515uOsb9PNVW7m874ztl+04GVH8qyVHqsuLU1Gnf/xuN6GHlbuYVHvTQn/0D9sD7cGxHaDJFawjU/X5Zg0Lha/+a2ovuGlbXm9qAzw7KlBJEufhBsn3JjRJqFbkgT0sr3HxyC/I1HSZEDHz5AsHLU60chT3huiqQUNtnD9SJ0XjucMluxAMgB9VZDksQI1wRGrbi3S5ow2a4sUNjId0n8hP3FfX/+ZvvmXLLCnZakerrLgE5UMR5U4VaownOpcoofdZOM1jVa2ZhO3455Ic7EsD51m+ckHotcVfLJ1oYkaTBKjSJCVlbtjGNBmGGQ6OmrsUI388MA/uvNg9W6pJ9BgeE/mXfFAMXD8dYLPGwjYjlBwUGApUWab8H0c1rCFfFe6d5yAqmmBmHYC2D3bq7Aa3NyaDhKbFhd0dABlddlFzggYommyt0KcEM/BbIxuW7xq0lTh6NpEO5ZnPVVmSoRuEO/R8dKbNfHQwRQRJWy4mmmQLTDPHVJ4I1Mb+zg55nj51lNAU6SCZRRy3/wyCLFD3Ui+LFWNs5HMW1zTh3+NKoEeVwb69Sm3D+uLDOxQA89kUaTT5txE27qt6AKoO3TsWpLkxmHLOInJRN54gTMeK0aoAWhtwvEFZphEynHenDYs9XmSsTPvCs8itta2WDj/fXttc23bYguMTeQbOkgAyAw0KiuSkiqrhg9u1fnYxhGn5KOGFTv6NzaE0z7WKolSRXxiYYHqB7i3QJmxBAT0HBR5nOX8inCBmTssqC3ifvKI9PWYoATAQzuMsA6omkbymbaAuCA4MqxJQjjyP2KdLIYvuxi94h0+cwtbeUy33mpPLENieOCqQoJLAPlklIDZwRjgo8oWcrvLcGMXDJoFPZq0l9oC6fdApXBocM3VXFVeYhQQeuI3s/vU4cXp7o5zXeLJbr7AfIKVfYGaG2bSWQLyBbESfluqIemQGf2Zq19+1Cgdm4KpqtTUMLg9exBOIf4pCrlzwkJYIZy/PD4l0D0X3xPzr45PyZeRuk78y5PWyxOGY3aBSHvHrRz1NIKXIh3AdKqMVAED0pIWPFqHEuHf6DHw8B49UQyJ2x2qZ4yzZb22IZbLxCcx7KfuKNdmt0iu+R//8qc/iMXz34VTfOiKc/y/Czk6kPfN08WTf5y/xH+LZ0/xPQx7oGS/xfPOexud5t2z1y/5VNwiwFI+D0elNmYVMpS6mELrhlG1ZfFtLr2uIrEA+zdPmf89Ihb+GGV5SpTMHu9+j5V0fwNBAvIEDVg+dItrKoscKO7zIUWaw3GwqlJR3+VMeXSpJ1v1YlUoPeM1ljXo6T7m/VKVuuXxlga6itY6PaTiZWxu5RoY2yLjeGcNKj3ketlX8IQLWIuj45SiA25wcdotLtmtb8qpMgnh3LJP09yr3onwfgLPuG8zZQ+Z0krCAZmcgoaPtAbSDIIlS9iSj2tJ09LOlNyHGgOJhrwHVM+1HRdiT0qqonOiswYoqVaz5BwJzqbOhsRcn81f4mi/JKdbfPbPb56/fgmKgO+/Qv1LNu7QV3qMJweInB7xmo3+65cdjplxTs0Pq3ishI0zWQ6JO8Mw9yULTuv061Ka6qjS1vLWB2BgdyhX7JC4tFKqZJmruBv0sfc9mQ4OIbpMw55bqxWNTG3nX3mpCf4skS6TGJA/kFxDUmpNE6dIclNIQRBueafrq2BNKJVqUq+j92cEGF2sj3MAYm5TkfYvYv4FxegDjlACHheiNI7Z3vwLQfPwitTXwRdW4wy6oNdF6P+68YRLiOP5K4QuAABLYZbntW3BKP8W1LWpqeyhXv3p6hHRl4f8qgQ904NBBhf8rrir0kxar9vqUNdn2VOClYjaS33YK9f5KkXSiq3VCLZubH1HjJAoQQpWkYUMdW5Y78jD2OQ2Sjl1HqVjoIQLQp8rbACxk7+l2kzJ3Bvf7a3EUmQT52tY7o1ZE4MHj3wUkiEeM8ldhzleCMiyePZi8ex56Ed0ria7R9516AgIfgSPbaZwWwQ0EMBHyrX8gEKJEnzVdBARj+B/hpomnBTP4eV504EKmnadZybb4ieWghByHFI56FpAUKev887C13/mKk9yKwcEp6/imBCYzjBGzbWHMUzJAO/0CX9YgHGe+FzFamVeEdjCzFAvutnMtsi4IM1rDtL29d76LApzV1KEZ2HOrond631z2pf0+JDanZayZDL2vIWtLjnHuFYY93jaJZ/NfCiMM3mOMRTTiXH68/me+qwNSvV6syw7rkJO7uH5ccUJD4scV4snL8U5/+GKC0u2n105zmZFQiwgDBXt51BhNJL5kMxOng/0Yu+nJJm4M1LPPbUIxfqz3WPWJ4TtvtWlftcE++lQQ+ejTGx062NSWURuRku8axrLJNTrM+OMXSpr9GBV6IKlnYqpJG8tKofw2lbPZmOGpX1ne0+w9oj+XOScvqm3eemesr3J91xVurLiabRtaLukVrQfChnHphXmIM21lHVx5TopLal32a51NVLPhyY8tbPLN1Cy3IkjxMPsZg9jnZDG25qS6U5LEZszjL1KEQUr4g4rwjNgWzRzy3Lx4gsM+PW31cs3UYXEXjXcK3KS9+mFUw2qSoZoByXzVy2P3pqZRirH4OavONpalgM7fcVJoj4Dbe8yFr3bg2jUO9Xx2DMEdkJb3Fl3bZAsGmkD99beMHXDnptmB8d4glJT22priNwp5r+nZhlS6vra+nYXCPBv9PH9dS65xT0EyQ827SHz27fu/mBjnVExUav9UslTSkRNk/hau6m4PZu/WsHaYAsUsxc4C9lj/vtLEbp1w/mrXSsUtaXs8q1notPQWzoOluE2IvE/bM3UJ1XnSaSHpSwQ7/amxmPyNXHQNPcsuR9gKSrIYV/Ca2c8VAAVN+AFk1Jnc6opvcfXPTVmN6Z1jTQhhTvPoLK4IsgxldWvm/Wu8+KWc1yblbY1PLuob6J2mpsqbhOEfO/D3y9nba+oPGL1S01qYGgw7Qy281ZU8bQh9y9bN2C+0LfS03lNQKBYn5bILfGJ+oGPTmK+dfaKZE51f98tZHtM0MWQrxb8OY9m8PUeQj5AQcSKspJE42oVTIYpN3uFLcTYG07JFj/OebvGlOzJLV1en7l5HGbvidnBzNIzYIDtZR9cit4jK8nixZeoIh+TOg89uCJ7Jcb3/6OxvSamTiq1WMIffv1leUZXvmBaP1KnX3+Zfv0lcq7th210EJwTWVKXRbh7N9ZUpq3TBsytSmouQyWkY4Wg/cReMLnGNQoVHC+3E6n98Pa1AqAU8MPiYJFeQ8EAAmUSBVC6BQfdaOX9mVXh/6uVR4JeoS4HIw2tgmAkh5KNvr6JkVaDDh38lWWZZNSlt0LRZZseD8lddmvoGOg0pUTy1oWvTUlJyfeUxDESA2pFcQiA5f7IcFy6i+h9FuUm+79MJ3JqvNvpVm+vTUfWiH0Fo+N9d4v0qLQf6Wibj23j/PgmPaEyjU5nFTz/nNIJqOxxLEI3nUDcDnad8cKK3de6ouvBIggPgK/kK0dJ34wSWNxfuIi4lDj7TZJhvbu+Trhuif6V+T/JwTHxijHKVYgZ96dlq8jh0/p1iPBx529ZZ11XZJJux9RxxcmwAdEUWzrf2P5Oq8Uf1NzZG5aKsLxK0oZWN352Y3bL0vJfh/PPSb+PxC8Wz56s4R+cyj5eWTx76lILvhzfJN7uBmHi337LIOtjH0WHCm71rusG2Evn0D0lzW5d79iOH/eR39IiPXXFq0/Yy7oJEPpyWCq1I3IIAKFQDVXUUqOy1zobAZbyj252aeAT6szYH1wAxYxaEz/lt/YCB+RyNC0oWEzCtmVqGUi6i5R0HUEE0W+cuXqex+AE3CvgnseAWKpK+Iofjs6FrM3EdDvihPheLTcRxoxcNslD97brX3b88VcNd+GLVcJT1S4MNtZnhZvNtwgbHi2SvfXLn11kl+IW8ciEUWRj8fTV4ul/ez75EXX5ilS1wpfz6CfMtXEQkGAuTTK2n2Es5CIL1b/1MSuQb5AykLYK4OCeTjOB1FasxnqSA2t0BNDD2JDekB9cv9FxwLq0nnGbFItnf8+6+AcOfPgZ/7WP/wmP6xv1saGd7fQA4jQa6s2K//0UE/7ypz/QOnuCic0JL5RcNqoLMwaKEwJT2wKgXb76L5pU3wvJlrKYukn+oQb9doV/L+IvVMkba6kau9m+LLO2NUE6QITUWqGmclXSL3ksMOu+7Cf1z03yKbmWIec98b8JcXei1lB1mjcoWPi2eaSc4eTyZZm976T872iW92ju/tbu7W9BKf5kM7qwFUFdC30Dusk/gcBmXHpfuKWhExz1FCAoRchfyfrX1jsNJdjYmM3/kxJfcazZEsWx4uKxV3+jphf+2ivPU3y82O+eXrKtji9u4mNDPWkRh4S6bwB/ZKUrrzot42kuWqQt7d3iXb90/XMuXgXV0lvLMDyQbuqfMrlC/hTq/j+XdCb05SYAAA=="}
W='/content/drive/MyDrive/geoproject/thesis_writeup/'
os.makedirs('/content/rn/tools',exist_ok=True)
for k,v in P.items(): open('/content/rn/'+k,'wb').write(gzip.decompress(base64.b64decode(v)))
EXPECT={'finalosmdraft.md':'9332d25214822c1dc01dbbeecceb264c','finalosmdraft_IEEE.md':'04d72548689bef4f67dbe73a526f5610'}
for f in ['finalosmdraft.md','finalosmdraft_IEEE.md']:
    print(f,'before',hashlib.md5(open(W+f,'rb').read()).hexdigest())
    shutil.copy(W+f,'/content/rn/'+f)
    r=subprocess.run(['python3','tools/renumber.py',f,'apa' if 'IEEE' not in f else 'ieee'],cwd='/content/rn',capture_output=True,text=True)
    print(r.stdout[-120:],r.stderr[-500:])
    h=hashlib.md5(open('/content/rn/'+f,'rb').read()).hexdigest(); print(f,'after',h,'MATCH' if h==EXPECT[f] else 'MISMATCH')
    if h==EXPECT[f]:
        shutil.copy(W+f, W+f.replace('.md','.v12.md')); shutil.copy('/content/rn/'+f, W+f); print('  written to Drive')

In [ ]:
import os, shutil, glob
W='/content/drive/MyDrive/geoproject/thesis_writeup/'
os.makedirs(W+'backups',exist_ok=True)
for f in glob.glob(W+'*.v12.md'): shutil.move(f, W+'backups/'+os.path.basename(f))
print(sorted(os.listdir(W)))

In [ ]:
import zipfile,re
for f in ['Thesis_APA.docx','Thesis_IEEE.docx']:
    xml=zipfile.ZipFile('/content/drive/MyDrive/geoproject/thesis_output/'+f).read('word/document.xml').decode()
    h=[re.sub('<[^>]+>','',p) for p in re.findall(r'<w:p[ >].*?</w:p>',xml,re.S) if re.search(r'w:pStyle w:val="Heading1"',p)]
    print(f,[x for x in h if x.strip()])
    eq=[re.sub('<[^>]+>','',p) for p in re.findall(r'<w:tr[ >].*?</w:tr>',xml,re.S) if '(3.1)' in p or '(3.22)' in p]
    print('  ',[e[:90] for e in eq])